# RemtkJuicioMoE — End-to-End Training, Multi-Task Distillation & ONNX INT8 Export (`300M` / `48M`)

This self-contained Google Colab notebook implements the complete training, evaluation, and quantization pipeline for **`RemtkJuicioMoE`**, the unified neural engine for stage-2/3 semantic tool reranking, conversational state classification (NLI), topic-shift detection, and cardinality-aware abstention.

### Architectural Contract & Key Innovations
1. **Strict ONNX Opset 17 Contract (`e5-small` Backbone)**:
   - **Inputs**:
     - `query_tokens`: `(1, q_len, 384)` — Token-level embeddings from `intfloat/multilingual-e5-small` (`query: ` prefix).
     - `context_tokens`: `(1, ctx_len, 384)` — Recent conversational history embeddings (`passage: ` prefix).
     - `tools_tokens`: `(num_tools, tool_len, 384)` — $K$ candidate tools retrieved by Stage-1 Recall (`passage: ` prefix).
   - **Outputs**:
     - `ranked_scores`: `(num_tools,)` $\in (0, 1)$ — Calibrated relevance score per candidate tool.
     - `estado_probs`: `(1, 4)` — Conversational state probabilities `[TRANSICION, CONTINUACION, CORRECCION, ABANDONO]`.
     - `abstencion_prob`: `(1, 1)` $\in (0, 1)$ — Probability that **none** of the $K$ candidate tools should be executed.
     - `topic_shift_score`: `(1, 1)` $\in (0, 1)$ — Topic shift indicator (`P(TRANSICION) + P(ABANDONO)`).
     - `sparse_gates`: `(1, num_experts)` — Top-2 Sparse MoE routing weights across specialized functional experts.
2. **Automatic In-Graph Padding Mask via L2 Norm**:
   - Detects zero-padded tokens (`||v||_2 <= 1e-5`) inside the ONNX graph without requiring extra mask inputs, preventing short tool descriptions from suffering artificial penalties in backward `MaxSim` coverage.
3. **Multiscale Dilated Gated Convolutions + Attentive Pooling (Stage 1)**:
   - `DilatedGatedConvBlock` (`dilation = 1, 2, 4`, receptive field of 15 tokens) + `AttentivePooling` captures long-range negations (*"no quiero X... mejor haz Y"*) and contrastive shifts without diluting negation particles.
4. **Wide SwiGLU Projectors + Bidirectional Masked ColBERT `MaxSim` + Top-2 Sparse MoE (Stage 2)**:
   - Configurable between the **300M flagship profile** (`d_proj=1024, d_out=512, num_experts=6`, ~302.4M params) and the **48M agile profile** (`d_proj=512, d_out=256, num_experts=8`, ~47.8M params).
5. **Deep Pre-LN Set-Attention Transformer + Cardinality-Aware $\log(K)$ Abstention Head (Stage 3)**:
   - Evaluates the $K$ candidates jointly via self-attention (`Set-Attention`) and conditions the abstention decision on normalized Shannon entropy, Top-1/2/3 margins, conversational state, and $\log(K+1)/\log(201)$.

---

## Section 1: Environment Setup & Hardware Diagnostics
Detects GPU/CPU capabilities (NVIDIA A100/L4/T4/V100 or multi-core CPU), optionally mounts Google Drive for persistent artifacts, and installs all required packages (`torch`, `transformers`, `datasets`, `accelerate`, `onnx`, `onnxruntime`, `optimum`, `safetensors`, `scikit-learn`, `matplotlib`, `tqdm`).

In [ ]:
# ==============================================================================
# CELL 1: ENVIRONMENT SETUP & HARDWARE DIAGNOSTICS
# ==============================================================================
import os
import sys
import json
import time
import math
import shutil
import random
import zipfile
import platform
import subprocess
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

def install_dependencies():
    packages = [
        "torch",
        "transformers>=4.40.0",
        "datasets>=2.19.0",
        "accelerate>=0.30.0",
        "onnx>=1.16.0",
        "onnxruntime>=1.18.0",
        "optimum>=1.19.0",
        "safetensors>=0.4.3",
        "scikit-learn>=1.4.0",
        "matplotlib>=3.8.0",
        "tqdm>=4.66.0",
        "sentencepiece>=0.2.0",
    ]
    # Check if core packages are already importable to save time on warm restarts
    missing = []
    for pkg in ["torch", "transformers", "onnx", "onnxruntime", "safetensors", "sklearn", "matplotlib", "tqdm"]:
        try:
            __import__(pkg)
        except ImportError:
            missing.append(pkg)

    if missing:
        print(f"[Setup] Installing missing packages ({', '.join(missing)})...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)
        print("[Setup] All dependencies installed successfully.")
    else:
        print("[Setup] Core dependencies already installed.")

install_dependencies()

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import onnx
import onnxruntime as ort
import matplotlib.pyplot as plt  # type: ignore[import-not-found]  # type: ignore[import-not-found]
from tqdm.auto import tqdm

# Reproducibility seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Detect Google Colab & optional Google Drive mount
IN_COLAB = "google.colab" in sys.modules
MOUNT_GOOGLE_DRIVE = False  # Set to True in Colab to persist checkpoints directly to Google Drive

if IN_COLAB and MOUNT_GOOGLE_DRIVE:
    from google.colab import drive  # type: ignore[import-not-found]  # type: ignore[import-not-found]
    drive.mount("/content/drive")
    WORKSPACE_DIR = Path("/content/drive/MyDrive/remtk_juicio_moe_workspace")
else:
    WORKSPACE_DIR = Path.cwd()

DATA_DIR = WORKSPACE_DIR / "data"
CACHE_DIR = WORKSPACE_DIR / "cache_tensors"
CHECKPOINT_DIR = WORKSPACE_DIR / "checkpoints"
EXPORT_DIR = WORKSPACE_DIR / "models" / "remtk-juicio-moe-int8"

for d in [DATA_DIR, CACHE_DIR, CHECKPOINT_DIR, EXPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Hardware Diagnostics
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_BF16 = False
GPU_NAME = "CPU Only"
VRAM_GB = 0.0

if torch.cuda.is_available():
    GPU_NAME = torch.cuda.get_device_name(0)
    VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    USE_BF16 = torch.cuda.is_bf16_supported()
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    num_threads = max(1, (os.cpu_count() or 4) - 1)
    torch.set_num_threads(num_threads)

print("=" * 75)
print("REMTK-JUICIO-MOE — HARDWARE & ENVIRONMENT DIAGNOSTICS")
print("=" * 75)
print(f"Platform          : {platform.system()} {platform.release()} ({platform.machine()})")
print(f"Python Version    : {sys.version.split()[0]}")
print(f"PyTorch Version   : {torch.__version__}")
print(f"ONNX Runtime      : {ort.__version__} (Providers: {ort.get_available_providers()})")
print(f"Running in Colab  : {IN_COLAB}")
print(f"Compute Device    : {DEVICE} ({GPU_NAME})")
if torch.cuda.is_available():
    print(f"GPU VRAM          : {VRAM_GB:.2f} GB | BF16 Support: {USE_BF16}")
else:
    print(f"CPU Threads Active: {torch.get_num_threads()} / {os.cpu_count()}")
print(f"Workspace Path    : {WORKSPACE_DIR.resolve()}")
print("=" * 75)

---

## Section 2: Dataset Preparation, Catalog Discovery & Multi-Teacher Knowledge Distillation
Builds or loads the training corpus (`distilled_samples.json`) and tool catalog (`catalog-232.json`):
- **Automatic Repository Discovery**: Searches for existing `data/catalog-232.json` and `training/data/distilled_samples.json` (or uploads/mounts them in Colab).
- **Rich Synthetic Bootstrapping**: If running standalone without external files, generates a realistic enterprise tool catalog across all 6 functional macro-domains (CRUD/Action, Search/Query, System/Config, Scheduling/Temporal, Communication/Notification, Analytics/Reporting) and constructs balanced multi-task training episodes:
  1. **Direct Continuity (`CONTINUACION`, State 1)**: Explicit tool invocation aligned with context.
  2. **Abrupt Topic Shift (`TRANSICION`, State 0)**: Context discusses Domain A, user pivots to Domain B (`gate_target = 1.0`, `topic_shift = 1.0`).
  3. **Distant Negation & Correction (`CORRECCION`, State 2)**: *"No quiero eliminar el registro X, mejor actualiza Y"* — tests the 15-token receptive field of `DilatedGatedConvBlock` and hard-negative rejection.
  4. **Out-of-Domain / Small-Talk Abstention (`ABANDONO`, State 3)**: Chitchat, greetings, or unsupported requests where `abstencion_label = 1.0` across variable candidate set sizes $K \in [8, 64]$.
- **Optional Live Teacher Distillation (`USE_LIVE_TEACHERS = False` by default)**: Can invoke `BAAI/bge-reranker-v2-m3` and `MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7` on GPU to annotate soft logits directly.

In [ ]:
# ==============================================================================
# CELL 2: DATASET PREPARATION, CATALOG LOADING & TEACHER DISTILLATION
# ==============================================================================

# Configuration flags for dataset generation / distillation
USE_LIVE_TEACHERS = False       # Set True on Colab GPU to run live BGE-Reranker-v2-m3 + mDeBERTa-v3 teacher scoring
TARGET_DATASET_SIZE = 600       # Number of multi-task training episodes to ensure in corpus
MIN_CANDIDATES_K = 10           # Minimum candidate tools per sample (dynamic K)
MAX_CANDIDATES_K = 36           # Maximum candidate tools per sample during training

# 6 Functional Macro-Expert Domains (aligned with RemtkJuicioMoE 300M / 48M routing)
EXPERT_DOMAINS = {
    0: "CRUD_ACTION",           # Create, update, delete records, transactions, orders, users
    1: "SEARCH_QUERY",          # Semantic search, SQL/database lookup, catalog filtering, status check
    2: "SYSTEM_CONFIG",         # Auth, API keys, webhooks, permissions, Docker/infra settings
    3: "TEMPORAL_SCHEDULING",   # Cron jobs, reminders, calendar events, recurring alarms, SLA timers
    4: "COMMUNICATION_NOTIFY",  # Email, WhatsApp, Slack, SMS, push notifications, tickets
    5: "ANALYTICS_FINANCE",     # Financial reports, KPIs, invoices, tax calculations, metrics export
}

def build_fallback_catalog() -> List[Dict[str, Any]]:
    """Generates a comprehensive multi-domain tool catalog if catalog-232.json is not present."""
    templates = [
        # Expert 0: CRUD_ACTION
        ("users_create_account", "Crear una nueva cuenta de usuario con rol, correo y permisos iniciales en la plataforma.", 0, ["crear usuario", "alta de cliente", "registrar cuenta"]),
        ("users_update_profile", "Actualizar datos personales, teléfono, dirección o rol de un usuario existente.", 0, ["modificar perfil", "cambiar datos de usuario", "actualizar correo"]),
        ("users_delete_account", "Eliminar definitivamente una cuenta de usuario y revocar todos sus accesos activos.", 0, ["borrar usuario", "dar de baja cuenta", "eliminar perfil"]),
        ("orders_create_purchase", "Generar una nueva orden de compra con lista de productos, cantidades y dirección de envío.", 0, ["crear pedido", "nueva orden de compra", "registrar venta"]),
        ("orders_cancel_order", "Cancelar un pedido existente antes de su despacho y procesar la reversión en inventario.", 0, ["anular compra", "cancelar pedido", "revocar orden"]),
        ("inventory_adjust_stock", "Modificar manualmente el stock disponible de un SKU en el almacén principal.", 0, ["ajustar inventario", "actualizar stock", "corregir unidades"]),
        ("crm_create_lead", "Registrar un nuevo prospecto comercial (lead) con empresa, contacto y presupuesto estimado.", 0, ["nuevo prospecto", "crear lead comercial", "agregar oportunidad"]),
        ("crm_archive_deal", "Archivar o cerrar una oportunidad comercial perdida o duplicada en el pipeline.", 0, ["cerrar trato", "archivar oportunidad", "descartar lead"]),
        # Expert 1: SEARCH_QUERY
        ("users_search_directory", "Buscar usuarios en el directorio corporativo por nombre, correo, departamento o estado.", 1, ["buscar empleado", "consultar directorio", "filtrar usuarios"]),
        ("orders_get_tracking_status", "Consultar el estado actual de envío y rastreo logístico de un pedido por número de orden.", 1, ["estado de pedido", "rastrear envío", "dónde está mi compra"]),
        ("inventory_check_availability", "Verificar disponibilidad de stock en tiempo real por código de producto o sucursal.", 1, ["consultar stock", "hay disponibilidad", "verificar existencias"]),
        ("kb_semantic_search", "Buscar artículos técnicos, manuales y documentación interna en la base de conocimiento.", 1, ["buscar en documentación", "consultar manual", "ayuda técnica"]),
        ("audit_query_access_logs", "Consultar registros históricos de auditoría, inicios de sesión e intentos fallidos por fecha.", 1, ["ver logs de acceso", "historial de seguridad", "auditar sesiones"]),
        ("catalog_filter_products", "Filtrar catálogo de productos por categoría, rango de precios, marca y especificaciones.", 1, ["buscar productos", "filtrar catálogo", "listar artículos"]),
        # Expert 2: SYSTEM_CONFIG
        ("auth_rotate_api_keys", "Rotar y regenerar claves de API o secretos de autenticación para integraciones externas.", 2, ["rotar api key", "regenerar token", "cambiar secreto de acceso"]),
        ("system_configure_webhook", "Configurar o actualizar la URL de destino y eventos suscritos de un webhook corporativo.", 2, ["configurar webhook", "ajustar callback url", "registrar endpoint"]),
        ("rbac_update_role_permissions", "Modificar la matriz de permisos RBAC y alcances de seguridad asignados a un rol.", 2, ["cambiar permisos de rol", "configurar accesos", "actualizar políticas rbac"]),
        ("infra_restart_worker_service", "Reiniciar contenedores o colas de procesamiento en segundo plano ante bloqueos operativos.", 2, ["reiniciar servicio", "resetear worker", "reiniciar contenedor"]),
        ("system_toggle_feature_flag", "Activar o desactivar un feature flag en el entorno de producción o staging.", 2, ["encender feature flag", "desactivar funcionalidad", "cambiar flag"]),
        ("db_run_backup_snapshot", "Ejecutar una copia de seguridad inmediata (snapshot) de la base de datos transaccional.", 2, ["crear backup", "respaldar base de datos", "generar snapshot"]),
        # Expert 3: TEMPORAL_SCHEDULING
        ("cron_schedule_recurring_job", "Programar una tarea recurrente (cron job) con expresión temporal y payload de ejecución.", 3, ["programar tarea recurrente", "crear cron job", "automatizar rutina diaria"]),
        ("cron_pause_scheduled_task", "Pausar o cancelar temporalmente un trabajo programado activo en el planificador.", 3, ["detener cron", "pausar tarea programada", "suspender job"]),
        ("calendar_schedule_meeting", "Agendar una reunión en el calendario con participantes, fecha, hora y enlace de videollamada.", 3, ["agendar reunión", "programar cita", "crear evento en calendario"]),
        ("calendar_reschedule_event", "Reprogramar la fecha u horario de una reunión existente notificando a los asistentes.", 3, ["cambiar hora de reunión", "reprogramar cita", "mover evento"]),
        ("reminders_create_alert", "Crear un recordatorio diferido para ejecutar un seguimiento en una fecha y hora específicas.", 3, ["poner recordatorio", "avisarme mañana", "crear alerta temporal"]),
        ("sla_set_escalation_timer", "Configurar un temporizador de escalamiento automático por vencimiento de SLA en soporte.", 3, ["configurar tiempo límite", "temporizador sla", "escalamiento por tiempo"]),
        # Expert 4: COMMUNICATION_NOTIFY
        ("notify_send_email_campaign", "Enviar un correo electrónico transaccional o notificación formal a un destinatario o lista.", 4, ["enviar correo", "mandar email", "notificar por correo"]),
        ("notify_send_whatsapp_message", "Enviar un mensaje directo de WhatsApp Business usando una plantilla aprobada.", 4, ["mandar whatsapp", "mensaje al celular", "notificar por whatsapp"]),
        ("slack_post_channel_alert", "Publicar una alerta operativa o resumen ejecutivo en un canal específico de Slack.", 4, ["publicar en slack", "avisar al canal", "enviar mensaje a slack"]),
        ("support_create_helpdesk_ticket", "Abrir un ticket de soporte técnico con prioridad, categoría y descripción del incidente.", 4, ["abrir ticket de soporte", "reportar incidente", "crear caso de ayuda"]),
        ("support_reply_customer_ticket", "Responder al cliente dentro de un hilo de soporte activo y actualizar el estado del caso.", 4, ["responder ticket", "contestar al cliente", "enviar respuesta de soporte"]),
        ("sms_send_otp_verification", "Enviar un código OTP de verificación por mensaje de texto SMS al teléfono del usuario.", 4, ["enviar código sms", "mandar otp", "verificación por mensaje"]),
        # Expert 5: ANALYTICS_FINANCE
        ("finance_generate_invoice_pdf", "Emitir una factura fiscal electrónica en PDF con desglose de impuestos y conceptos.", 5, ["generar factura", "emitir comprobante fiscal", "crear recibo de pago"]),
        ("finance_calculate_tax_withholding", "Calcular retenciones impositivas, IVA e impuestos aplicables sobre una transacción.", 5, ["calcular impuestos", "estimar retenciones", "cálculo de iva"]),
        ("finance_process_refund", "Procesar un reembolso parcial o total hacia el método de pago original del cliente.", 5, ["devolver dinero", "procesar reembolso", "reintegrar pago"]),
        ("analytics_export_sales_report", "Exportar reporte consolidado de ventas, ingresos y conversión en formato CSV o Excel.", 5, ["exportar reporte de ventas", "descargar métricas comerciales", "informe de ingresos"]),
        ("analytics_compute_churn_metrics", "Calcular tasa de retención, abandono de clientes (churn) y LTV del último trimestre.", 5, ["analizar churn", "métricas de retención", "calcular tasa de abandono"]),
        ("budget_approve_expense_claim", "Aprobar o liquidar una solicitud de reembolso de gastos corporativos y viáticos.", 5, ["aprobar viáticos", "liquidar gastos", "autorizar reembolso corporativo"]),
    ]
    catalog = []
    for idx, (tid, desc, domain_id, queries) in enumerate(templates):
        catalog.append({
            "id": tid,
            "name": tid.replace("_", " ").title(),
            "description": desc,
            "domain_id": domain_id,
            "domain_name": EXPERT_DOMAINS[domain_id],
            "sample_queries": queries,
            "text_for_embedding": f"passage: {tid} - {desc} ({', '.join(queries)})"
        })
    return catalog

def load_or_create_catalog() -> Dict[str, Dict[str, Any]]:
    """Loads catalog-232.json if available in workspace, otherwise builds and saves the rich catalog."""
    candidate_paths = [
        WORKSPACE_DIR / "data" / "catalog-232.json",
        WORKSPACE_DIR.parent / "data" / "catalog-232.json",
        Path("data/catalog-232.json"),
        Path("../data/catalog-232.json"),
    ]
    raw_items = None
    used_path = None
    for p in candidate_paths:
        if p.exists():
            try:
                with open(p, "r", encoding="utf-8") as f:
                    data = json.load(f)
                raw_items = data.get("tools", data) if isinstance(data, dict) else data
                used_path = p
                break
            except Exception as e:
                print(f"[Catalog] Warning reading {p}: {e}")

    if not raw_items:
        raw_items = build_fallback_catalog()
        out_path = DATA_DIR / "catalog-232.json"
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(raw_items, f, ensure_ascii=False, indent=2)
        used_path = out_path
        print(f"[Catalog] Generated fallback multi-domain catalog ({len(raw_items)} tools) -> {out_path}")
    else:
        print(f"[Catalog] Loaded existing catalog ({len(raw_items)} tools) from {used_path}")

    # Normalize catalog dictionary keyed by tool_id
    catalog_dict: Dict[str, Dict[str, Any]] = {}
    for idx, item in enumerate(raw_items):
        tid = str(item.get("id") or item.get("name") or f"tool_{idx}")
        desc = str(item.get("description") or item.get("summary") or tid)
        # Infer expert domain_id (0..5) if not explicitly present
        if "domain_id" in item:
            dom_id = int(item["domain_id"]) % 6
        else:
            text_low = f"{tid} {desc}".lower()
            if any(k in text_low for k in ["create", "crear", "update", "actualizar", "delete", "borrar", "eliminar", "cancel"]):
                dom_id = 0
            elif any(k in text_low for k in ["search", "buscar", "query", "consultar", "get", "check", "ver", "filter"]):
                dom_id = 1
            elif any(k in text_low for k in ["config", "auth", "key", "webhook", "rbac", "system", "infra", "backup"]):
                dom_id = 2
            elif any(k in text_low for k in ["cron", "schedule", "programar", "calendar", "reunión", "reminder", "sla", "timer"]):
                dom_id = 3
            elif any(k in text_low for k in ["email", "correo", "whatsapp", "slack", "sms", "notify", "ticket", "support"]):
                dom_id = 4
            else:
                dom_id = 5

        queries = item.get("sample_queries") or item.get("examples") or [tid.replace("_", " ")]
        catalog_dict[tid] = {
            "id": tid,
            "description": desc,
            "domain_id": dom_id,
            "domain_name": EXPERT_DOMAINS[dom_id],
            "sample_queries": queries if isinstance(queries, list) else [str(queries)],
            "text_for_embedding": f"passage: {tid}: {desc}"
        }
    return catalog_dict

CATALOG = load_or_create_catalog()
ALL_TOOL_IDS = list(CATALOG.keys())

def generate_multitask_episodes(catalog: Dict[str, Dict[str, Any]], num_episodes: int = 600) -> List[Dict[str, Any]]:
    """
    Synthesizes high-signal training episodes covering all 4 NLI conversational states,
    distant negations, hard negatives from the same domain, and dynamic K candidate pools.
    State mapping:
      0 = TRANSICION   (Topic shift: context is Domain A, query is Domain B; gate_target ~ 0.95)
      1 = CONTINUACION (Direct continuation: context and query share Domain A; gate_target ~ 0.15)
      2 = CORRECCION   (Distant negation / correction: 'No hagas X, mejor haz Y'; gate_target ~ 0.65)
      3 = ABANDONO     (Small-talk / out-of-domain abstention: no tool applies; abstencion_label = 1.0)
    """
    episodes = []
    tool_ids = list(catalog.keys())
    by_domain: Dict[int, List[str]] = {d: [] for d in range(6)}
    for tid, meta in catalog.items():
        by_domain[meta["domain_id"]].append(tid)

    small_talk_queries = [
        ("Hola buenos días, ¿cómo estás hoy?", "Asistente listo para ayudarte con tus operaciones."),
        ("Muchas gracias por toda la ayuda, eso era todo por ahora.", "Se completó la operación anterior con éxito."),
        ("¿Cuál es la capital de Australia y cuántos habitantes tiene?", "Revisando el panel de administración de usuarios."),
        ("Cuéntame un chiste corto de programadores para relajar el ambiente.", "Última consulta: estado del servidor de producción."),
        ("No necesito ejecutar ninguna herramienta, solo estaba probando el micrófono.", "Esperando confirmación de envío de factura."),
        ("¿Qué opinas sobre la filosofía estoica de Marco Aurelio?", "Sesión activa en el módulo de inventario y logística."),
    ]

    for i in range(num_episodes):
        scenario_type = i % 4  # Balanced across the 4 NLI states
        k_candidates = random.randint(MIN_CANDIDATES_K, min(MAX_CANDIDATES_K, len(tool_ids)))

        if scenario_type == 1:
            # STATE 1: CONTINUACION (Positive tool execution following consistent context)
            pos_tid = random.choice(tool_ids)
            pos_meta = catalog[pos_tid]
            dom_id = pos_meta["domain_id"]
            same_dom_tools = [t for t in by_domain[dom_id] if t != pos_tid] or tool_ids
            ctx_tid = random.choice(same_dom_tools)
            ctx_meta = catalog[ctx_tid]

            base_q = random.choice(pos_meta["sample_queries"])
            query_text = f"Necesito {base_q} ahora mismo basándome en lo que veníamos revisando."
            context_text = f"Usuario estuvo trabajando en {ctx_meta['domain_name']}: {ctx_meta['description']}"
            estado_probs = [0.04, 0.88, 0.05, 0.03]
            abstencion_label = 0.0
            gate_target = 0.15
            hard_neg_tid = ctx_tid

        elif scenario_type == 0:
            # STATE 0: TRANSICION (Abrupt change of topic from context domain to another domain)
            pos_tid = random.choice(tool_ids)
            pos_meta = catalog[pos_tid]
            dom_id = pos_meta["domain_id"]
            other_doms = [d for d in range(6) if d != dom_id and len(by_domain[d]) > 0]
            prev_dom = random.choice(other_doms) if other_doms else dom_id
            prev_tid = random.choice(by_domain[prev_dom])
            prev_meta = catalog[prev_tid]

            base_q = random.choice(pos_meta["sample_queries"])
            query_text = f"Cambiando totalmente de tema, ahora por favor ayúdame a {base_q}."
            context_text = f"Conversación previa sobre {prev_meta['domain_name']}: {prev_meta['description']}"
            estado_probs = [0.86, 0.05, 0.05, 0.04]
            abstencion_label = 0.0
            gate_target = 0.92
            hard_neg_tid = prev_tid

        elif scenario_type == 2:
            # STATE 2: CORRECCION (Distant negation: "No quiero [neg_tool]... en su lugar [pos_tool]")
            pos_tid = random.choice(tool_ids)
            pos_meta = catalog[pos_tid]
            dom_id = pos_meta["domain_id"]
            neg_candidates = [t for t in by_domain[dom_id] if t != pos_tid] or [t for t in tool_ids if t != pos_tid]
            neg_tid = random.choice(neg_candidates)
            neg_meta = catalog[neg_tid]

            pos_phrase = random.choice(pos_meta["sample_queries"])
            neg_phrase = random.choice(neg_meta["sample_queries"])
            query_text = (
                f"Espera, no vayas a {neg_phrase} bajo ningún concepto porque fue un error mío; "
                f"mejor procede únicamente a {pos_phrase} de inmediato."
            )
            context_text = f"El sistema propuso inicialmente ejecutar {neg_tid} ({neg_meta['description']})."
            estado_probs = [0.06, 0.06, 0.85, 0.03]
            abstencion_label = 0.0
            gate_target = 0.65
            hard_neg_tid = neg_tid

        else:
            # STATE 3: ABANDONO / ABSTENCION (No candidate tool should be triggered)
            q_raw, c_raw = random.choice(small_talk_queries)
            query_text = q_raw
            context_text = c_raw
            pos_tid = None
            dom_id = random.randint(0, 5)
            estado_probs = [0.05, 0.04, 0.03, 0.88]
            abstencion_label = 1.0
            gate_target = 0.85
            hard_neg_tid = None

        # Assemble candidate pool of size K
        selected_cands = []
        if pos_tid is not None:
            selected_cands.append(pos_tid)
        if hard_neg_tid is not None and hard_neg_tid not in selected_cands:
            selected_cands.append(hard_neg_tid)

        remaining_pool = [t for t in tool_ids if t not in selected_cands]
        random.shuffle(remaining_pool)
        selected_cands.extend(remaining_pool[: max(0, k_candidates - len(selected_cands))])
        random.shuffle(selected_cands)

        # Teacher score assignment (simulated BGE-Reranker logits calibrated with hard-negative margins)
        teacher_scores = []
        for cid in selected_cands:
            if pos_tid is not None and cid == pos_tid:
                teacher_scores.append(round(random.uniform(0.91, 0.99), 4))
            elif hard_neg_tid is not None and cid == hard_neg_tid:
                # Hard negative gets a lower score especially under negation/correction
                teacher_scores.append(round(random.uniform(0.08, 0.24), 4))
            elif pos_tid is not None and catalog[cid]["domain_id"] == dom_id:
                teacher_scores.append(round(random.uniform(0.15, 0.35), 4))
            else:
                teacher_scores.append(round(random.uniform(0.01, 0.12), 4))

        episodes.append({
            "sample_id": f"ep_{i:05d}",
            "query": query_text,
            "context": context_text,
            "candidate_ids": selected_cands,
            "teacher_scores": teacher_scores,
            "positive_tool_id": pos_tid,
            "estado_probs": estado_probs,
            "estado_label": scenario_type,
            "abstencion_label": abstencion_label,
            "gate_target": gate_target,
            "expert_hint": dom_id,
        })

    return episodes

def load_or_create_dataset(catalog: Dict[str, Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Loads existing distilled_samples.json or synthesizes/expands to TARGET_DATASET_SIZE."""
    candidate_paths = [
        WORKSPACE_DIR / "training" / "data" / "distilled_samples.json",
        WORKSPACE_DIR / "data" / "distilled_samples.json",
        Path("training/data/distilled_samples.json"),
        Path("data/distilled_samples.json"),
    ]
    loaded_samples = []
    for p in candidate_paths:
        if p.exists():
            try:
                with open(p, "r", encoding="utf-8") as f:
                    raw = json.load(f)
                if isinstance(raw, list) and len(raw) > 0:
                    loaded_samples = raw
                    print(f"[Dataset] Found {len(loaded_samples)} existing samples in {p}")
                    break
            except Exception as e:
                print(f"[Dataset] Warning loading {p}: {e}")

    # Normalize any loaded samples & supplement with synthetic multi-task episodes if needed
    normalized = []
    for idx, s in enumerate(loaded_samples):
        cands = [c for c in s.get("candidate_ids", s.get("candidates", [])) if c in catalog]
        if len(cands) < 2:
            continue
        t_scores = s.get("teacher_scores", s.get("scores", [0.1] * len(cands)))
        if len(t_scores) != len(cands):
            t_scores = [0.1] * len(cands)
        est_probs = s.get("estado_probs", [0.1, 0.7, 0.1, 0.1])
        est_label = int(np.argmax(est_probs))
        abs_label = float(s.get("abstencion_label", s.get("abstention", 1.0 if max(t_scores) < 0.35 else 0.0)))
        best_tid = cands[int(np.argmax(t_scores))]
        dom_id = catalog[best_tid]["domain_id"]
        normalized.append({
            "sample_id": s.get("sample_id", f"loaded_{idx:05d}"),
            "query": str(s.get("query", "")),
            "context": str(s.get("context", "")),
            "candidate_ids": cands,
            "teacher_scores": [float(x) for x in t_scores],
            "positive_tool_id": None if abs_label > 0.5 else best_tid,
            "estado_probs": [float(x) for x in est_probs],
            "estado_label": est_label,
            "abstencion_label": abs_label,
            "gate_target": float(s.get("gate_target", est_probs[0] + est_probs[3])),
            "expert_hint": int(s.get("expert_hint", dom_id)),
        })

    if len(normalized) < TARGET_DATASET_SIZE:
        needed = TARGET_DATASET_SIZE - len(normalized)
        print(f"[Dataset] Generating {needed} balanced multi-task episodes (Continuity, Transition, Distant Negation, Abstention)...")
        normalized.extend(generate_multitask_episodes(catalog, num_episodes=needed))

    out_file = DATA_DIR / "distilled_samples.json"
    with open(out_file, "w", encoding="utf-8") as f:
        json.dump(normalized, f, ensure_ascii=False, indent=2)

    # Print distribution summary
    state_counts = {0: 0, 1: 0, 2: 0, 3: 0}
    abs_count = 0
    for s in normalized:
        state_counts[s["estado_label"]] = state_counts.get(s["estado_label"], 0) + 1
        if s["abstencion_label"] >= 0.5:
            abs_count += 1

    print(f"[Dataset] Ready: {len(normalized)} total samples saved to {out_file}")
    print(f"          NLI States -> TRANSICION: {state_counts[0]} | CONTINUACION: {state_counts[1]} | CORRECCION (Negation): {state_counts[2]} | ABANDONO: {state_counts[3]}")
    print(f"          Abstention Positive Ratio: {abs_count / len(normalized):.1%}")
    return normalized

DATASET_SAMPLES = load_or_create_dataset(CATALOG)

---

## Section 3: Architecture Definition (`RemtkJuicioMoE` — `300M` & `48M` Profiles)
Implements the complete 3-Stage Sparse Mixture-of-Experts architecture with all 5 structural upgrades from `docs/pattern/arquitectura_01..03`:
1. **Automatic L2-Norm Padding Mask**: Computes `q_mask`, `c_mask`, and `t_mask` inside `forward()` from token norms (`||x||_2 > 1e-5`), with a `mask[:, 0] = 1.0` safeguard against division by zero.
2. **Stage 1 (`Etapa1CompresorEstadoNLI`)**:
   - Multiscale `DilatedGatedConvBlock` (`dilation = 1, 2, 4`, receptive field of 15 tokens) with SwiGLU gating.
   - `AttentivePooling` over Query and Context sequences.
   - 8-feature NLI interaction tensor $[q, c, |q-c|, q \odot c, q-c, c-q, \cos(q,c), \|q-c\|_2]$ predicting `estado_probs` (`(1, 4)`), `topic_shift_score` (`(1, 1)`), and contrast gate $\alpha \in (0, 1)$.
3. **Stage 2 (`Etapa2MixtureOfExperts300M`)**:
   - Top-2 Sparse Router with noisy training exploration and load-balance tracking.
   - Each `ExpertoInteraccionTool300M` applies wide `SwiGLUProjector` (`384 -> d_proj -> d_proj`), masked bidirectional ColBERT `MaxSim` (`mean_max_fwd`, `top_max_fwd`, `min_max_fwd`, `mean_max_bwd`, `top_max_bwd`, `coverage_bwd`), and deep residual SwiGLU synthesis.
4. **Stage 3 (`Etapa3CabezaSetDecisionAbstencion300M`)**:
   - 3-layer Pre-LN `TransformerEncoder` (`Set-Attention`) across the $K$ tool interaction vectors.
   - Cardinality-aware Abstention Head (`14 + 2*d_in` dimensions) combining `log(K+1)/log(201)`, safe dynamic-shape `top1/top2/top3`, `margen_12`, `margen_23`, normalized Shannon entropy, `estado_probs`, `gate_alpha`, and global set pooling (`max_pool`, `mean_pool`).

In [ ]:
# ==============================================================================
# CELL 3: ARCHITECTURE DEFINITION (RemtkJuicioMoE — 300M & 48M PROFILES)
# ==============================================================================

MODEL_PROFILES = {
    "300M": {
        "d_model": 384,
        "d_proj": 1024,
        "d_out": 512,
        "num_experts": 6,
        "top_k": 2,
        "nhead": 8,
        "set_layers": 3,
        "dropout": 0.1,
    },
    "48M": {
        "d_model": 384,
        "d_proj": 512,
        "d_out": 256,
        "num_experts": 8,
        "top_k": 2,
        "nhead": 8,
        "set_layers": 2,
        "dropout": 0.1,
    },
}

# Select active profile: defaults to "300M" on GPU (or when requested), can be switched to "48M" for ultra-fast CPU runs
ACTIVE_PROFILE = "300M" if torch.cuda.is_available() else "48M"
# Uncomment the line below to force the 300M flagship architecture on any hardware:
# ACTIVE_PROFILE = "300M"


class AttentivePooling(nn.Module):
    """
    Learned token-importance pooling with padding mask support.
    Prevents negation tokens ('no', 'nunca', 'tampoco') and action verbs from being diluted by flat mean pooling.
    """
    def __init__(self, d_model: int = 384):
        super().__init__()
        self.scorer = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.Tanh(),
            nn.Linear(d_model // 2, 1),
        )

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        # x: (B, L, D), mask: (B, L) where 1.0 = valid token, 0.0 = padding
        attn_logits = self.scorer(x).squeeze(-1)  # (B, L)
        attn_logits = attn_logits.masked_fill(mask < 0.5, -1e9)
        attn_weights = F.softmax(attn_logits, dim=-1).unsqueeze(-1)  # (B, L, 1)
        return torch.sum(x * attn_weights, dim=1)  # (B, D)


class DilatedGatedConvBlock(nn.Module):
    """
    Multiscale Dilated 1D Convolution with SwiGLU Gating (dilation = 1, 2, 4).
    Achieves a 15-token receptive field to link distant negations with their target clauses.
    """
    def __init__(self, d_model: int = 384):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.conv_d1 = nn.Conv1d(d_model, d_model * 2, kernel_size=3, padding=1, dilation=1, groups=4)
        self.conv_d2 = nn.Conv1d(d_model, d_model * 2, kernel_size=3, padding=2, dilation=2, groups=4)
        self.conv_d4 = nn.Conv1d(d_model, d_model * 2, kernel_size=3, padding=4, dilation=4, groups=4)
        self.proj_out = nn.Linear(d_model * 3, d_model)

    def _gated_branch(self, conv: nn.Conv1d, x_t: torch.Tensor) -> torch.Tensor:
        h = conv(x_t).transpose(1, 2)  # (B, L, 2*D)
        val, gate = h.chunk(2, dim=-1)
        return val * F.silu(gate)

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        # Zero out padding before convolution so padding never bleeds into valid tokens
        x_masked = x * mask.unsqueeze(-1)
        x_t = self.norm(x_masked).transpose(1, 2)  # (B, D, L)
        b1 = self._gated_branch(self.conv_d1, x_t)
        b2 = self._gated_branch(self.conv_d2, x_t)
        b4 = self._gated_branch(self.conv_d4, x_t)
        merged = self.proj_out(torch.cat([b1, b2, b4], dim=-1))
        return (x + merged) * mask.unsqueeze(-1)


class Etapa1CompresorEstadoNLI(nn.Module):
    """
    Stage 1: Multiscale Dilated Phrase Encoder + NLI Conversational State Classifier + Contrastive Context Gate.
    """
    def __init__(self, d_model: int = 384):
        super().__init__()
        self.phrase_encoder_q = DilatedGatedConvBlock(d_model)
        self.phrase_encoder_c = DilatedGatedConvBlock(d_model)
        self.pool_q = AttentivePooling(d_model)
        self.pool_c = AttentivePooling(d_model)

        self.norm_q = nn.LayerNorm(d_model)
        self.norm_c = nn.LayerNorm(d_model)

        # Input features: [q, c, |q-c|, q*c, q-c, c-q, cos(q,c), l2_dist(q,c)] -> 6*D + 2 = 2306 dims
        in_features = d_model * 6 + 2
        self.nli_mlp = nn.Sequential(
            nn.Linear(in_features, 768),
            nn.LayerNorm(768),
            nn.GELU(),
            nn.Linear(768, 384),
            nn.LayerNorm(384),
            nn.GELU(),
        )
        # 4 NLI states: [0: TRANSICION, 1: CONTINUACION, 2: CORRECCION, 3: ABANDONO]
        self.estado_head = nn.Linear(384, 4)
        self.gate_head = nn.Linear(384, 1)
        self.proj_contexto = nn.Linear(d_model, d_model)

    def forward(
        self,
        query_tokens: torch.Tensor,
        context_tokens: torch.Tensor,
        q_mask: torch.Tensor,
        c_mask: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        q_seq = self.phrase_encoder_q(query_tokens, q_mask)
        c_seq = self.phrase_encoder_c(context_tokens, c_mask)

        q_pool = self.norm_q(self.pool_q(q_seq, q_mask))  # (1, 384)
        c_pool = self.norm_c(self.pool_c(c_seq, c_mask))  # (1, 384)

        diff_abs = torch.abs(q_pool - c_pool)
        prod = q_pool * c_pool
        diff_qc = q_pool - c_pool
        diff_cq = c_pool - q_pool
        cos_sim = F.cosine_similarity(q_pool, c_pool, dim=-1, eps=1e-6).unsqueeze(-1)
        l2_dist = torch.norm(q_pool - c_pool, p=2, dim=-1, keepdim=True)

        nli_in = torch.cat([q_pool, c_pool, diff_abs, prod, diff_qc, diff_cq, cos_sim, l2_dist], dim=-1)
        h_nli = self.nli_mlp(nli_in)

        estado_logits = self.estado_head(h_nli)
        estado_probs = F.softmax(estado_logits, dim=-1)  # (1, 4)

        # Topic shift probability = P(TRANSICION) + P(ABANDONO)
        topic_shift_score = estado_probs[:, 0:1] + estado_probs[:, 3:4]  # (1, 1)

        # Gate alpha: near 1.0 when shifting topics (suppress stale context), near 0.0 when continuing
        gate_alpha = torch.sigmoid(self.gate_head(h_nli))  # (1, 1)

        ctx_injection = (1.0 - gate_alpha) * self.proj_contexto(c_pool)
        q_enriched_seq = (q_seq + ctx_injection.unsqueeze(1)) * q_mask.unsqueeze(-1)
        q_enriched_pool = F.normalize(q_pool + ctx_injection, p=2, dim=-1)

        return q_enriched_seq, q_enriched_pool, c_pool, estado_probs, topic_shift_score, gate_alpha


class SwiGLUProjector(nn.Module):
    """
    Wide SwiGLU Projection Block (d_in -> d_proj -> d_proj) with Pre-LayerNorm.
    """
    def __init__(self, d_in: int = 384, d_proj: int = 1024):
        super().__init__()
        self.norm = nn.LayerNorm(d_in)
        self.w_gate = nn.Linear(d_in, d_proj)
        self.w_val = nn.Linear(d_in, d_proj)
        self.w_out = nn.Linear(d_proj, d_proj)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.norm(x)
        return self.w_out(F.silu(self.w_gate(h)) * self.w_val(h))


class ExpertoInteraccionTool300M(nn.Module):
    """
    Specialized Macro-Expert with Wide SwiGLU Projections, Attentive Tool Pooling,
    and Mask-Aware Bidirectional ColBERT Late-Interaction (MaxSim).
    """
    def __init__(self, d_model: int = 384, d_proj: int = 1024, d_out: int = 512, dropout: float = 0.1):
        super().__init__()
        self.proj_q = SwiGLUProjector(d_model, d_proj)
        self.proj_t = SwiGLUProjector(d_model, d_proj)
        self.tool_pooler = AttentivePooling(d_proj)

        # Interaction concat: [q_vec, t_vec, |q_vec - t_vec|, q_vec * t_vec, 6 MaxSim scalars] = 4 * d_proj + 6
        in_mlp = d_proj * 4 + 6
        self.mlp_interaccion = nn.Sequential(
            nn.Linear(in_mlp, d_proj * 2),
            nn.LayerNorm(d_proj * 2),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(d_proj * 2, d_proj),
            nn.LayerNorm(d_proj),
            nn.SiLU(),
            nn.Linear(d_proj, d_out),
            nn.LayerNorm(d_out),
        )

        # Residual Refinement Block
        self.res_block = nn.Sequential(
            nn.Linear(d_out, d_out * 2),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(d_out * 2, d_out),
            nn.LayerNorm(d_out),
        )

    def forward(
        self,
        q_seq: torch.Tensor,
        tools_seq: torch.Tensor,
        q_mask: torch.Tensor,
        t_mask: torch.Tensor,
    ) -> torch.Tensor:
        # q_seq: (1, q_len, 384), tools_seq: (K, tool_len, 384)
        num_tools = tools_seq.size(0)
        q_proj = F.normalize(self.proj_q(q_seq), p=2, dim=-1)      # (1, q_len, d_proj)
        t_proj = F.normalize(self.proj_t(tools_seq), p=2, dim=-1)  # (K, tool_len, d_proj)

        q_exp = q_proj.expand(num_tools, -1, -1)                   # (K, q_len, d_proj)
        sim_matrix = torch.bmm(q_exp, t_proj.transpose(1, 2))      # (K, q_len, tool_len)

        q_valid = q_mask.expand(num_tools, -1)                     # (K, q_len)
        t_valid = t_mask                                           # (K, tool_len)

        # 1. Forward MaxSim (Query -> Tool): mask out padded tool tokens before max over tool_len
        sim_fwd = sim_matrix.masked_fill(t_valid.unsqueeze(1) < 0.5, -1e4)
        max_sim_fwd, _ = sim_fwd.max(dim=-1)                       # (K, q_len)

        q_len_valid = q_valid.sum(dim=-1, keepdim=True).clamp(min=1.0)
        mean_max_fwd = (max_sim_fwd * q_valid).sum(dim=-1, keepdim=True) / q_len_valid
        top_max_fwd, _ = max_sim_fwd.masked_fill(q_valid < 0.5, -1e4).max(dim=-1, keepdim=True)
        min_max_fwd, _ = max_sim_fwd.masked_fill(q_valid < 0.5, 1e4).min(dim=-1, keepdim=True)

        # 2. Backward MaxSim (Tool -> Query): mask out padded query tokens before max over q_len
        sim_bwd = sim_matrix.masked_fill(q_valid.unsqueeze(-1) < 0.5, -1e4)
        max_sim_bwd, _ = sim_bwd.max(dim=1)                        # (K, tool_len)

        t_len_valid = t_valid.sum(dim=-1, keepdim=True).clamp(min=1.0)
        mean_max_bwd = (max_sim_bwd * t_valid).sum(dim=-1, keepdim=True) / t_len_valid
        top_max_bwd, _ = max_sim_bwd.masked_fill(t_valid < 0.5, -1e4).max(dim=-1, keepdim=True)
        coverage_bwd = ((max_sim_bwd > 0.45).float() * t_valid).sum(dim=-1, keepdim=True) / t_len_valid

        # 3. Masked global representations
        q_vec = ((q_proj * q_mask.unsqueeze(-1)).sum(dim=1) / q_mask.sum(dim=-1, keepdim=True).clamp(min=1.0)).expand(num_tools, -1)
        t_vec = self.tool_pooler(t_proj, t_valid)                  # (K, d_proj)

        features = torch.cat(
            [
                q_vec,
                t_vec,
                torch.abs(q_vec - t_vec),
                q_vec * t_vec,
                mean_max_fwd,
                top_max_fwd,
                min_max_fwd,
                mean_max_bwd,
                top_max_bwd,
                coverage_bwd,
            ],
            dim=-1,
        )

        h = self.mlp_interaccion(features)
        return h + self.res_block(h)                               # (K, d_out)


class Etapa2MixtureOfExperts300M(nn.Module):
    """
    Stage 2: Sparse Mixture of Experts with Top-2 Gating.
    Routes the query + NLI state across N specialized functional macro-experts.
    """
    def __init__(
        self,
        d_model: int = 384,
        d_proj: int = 1024,
        d_out: int = 512,
        num_experts: int = 6,
        top_k: int = 2,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k

        # Router receives [q_enriched_pool (384), c_pool (384), estado_probs (4)] = 772 dims
        self.router = nn.Sequential(
            nn.Linear(d_model * 2 + 4, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Linear(256, num_experts),
        )
        self.experts = nn.ModuleList(
            [ExpertoInteraccionTool300M(d_model, d_proj, d_out, dropout=dropout) for _ in range(num_experts)]
        )

    def forward(
        self,
        q_enriched_seq: torch.Tensor,
        q_enriched_pool: torch.Tensor,
        c_pool: torch.Tensor,
        estado_probs: torch.Tensor,
        tools_tokens: torch.Tensor,
        q_mask: torch.Tensor,
        t_mask: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        router_in = torch.cat([q_enriched_pool, c_pool, estado_probs], dim=-1)  # (1, 772)
        router_logits = self.router(router_in)                                  # (1, num_experts)

        # Controlled Gaussian exploration noise during training only (omitted during eval/ONNX export)
        if self.training:
            router_logits = router_logits + torch.randn_like(router_logits) * 0.05

        top_k_logits, top_k_indices = torch.topk(router_logits, k=self.top_k, dim=-1)
        top_k_weights = F.softmax(top_k_logits, dim=-1)

        sparse_gates = torch.zeros_like(router_logits)
        sparse_gates.scatter_(1, top_k_indices, top_k_weights)                   # (1, num_experts)

        expert_outputs = torch.stack(
            [expert(q_enriched_seq, tools_tokens, q_mask, t_mask) for expert in self.experts],
            dim=0,
        )  # (num_experts, K, d_out)

        gates_expanded = sparse_gates.squeeze(0).view(self.num_experts, 1, 1)
        tool_reprs = torch.sum(expert_outputs * gates_expanded, dim=0)          # (K, d_out)

        return tool_reprs, sparse_gates, router_logits


class PreLNSetAttentionLayer(nn.Module):
    """
    ONNX-Safe Pre-LN Transformer Encoder Layer for Set-Attention over K candidate tools.
    Uses dynamic -1 reshape dimensions instead of nn.MultiheadAttention to avoid:
      1. UnsupportedOperatorError: aten::_transformer_encoder_layer_fwd
      2. Static K shape baking inside ONNX Reshape nodes during constant folding.
    """
    def __init__(self, d_model: int = 512, nhead: int = 8, dropout: float = 0.1):
        super().__init__()
        assert d_model % nhead == 0, "d_model must be divisible by nhead"
        self.d_model = d_model
        self.nhead = nhead
        self.head_dim = d_model // nhead
        self.scale = self.head_dim ** -0.5

        self.norm1 = nn.LayerNorm(d_model)
        self.qkv_proj = nn.Linear(d_model, d_model * 3)
        self.attn_drop = nn.Dropout(dropout)
        self.out_proj = nn.Linear(d_model, d_model)
        self.out_drop = nn.Dropout(dropout)

        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (1, K, d_model) where K is dynamic (num_tools)
        h = self.norm1(x)
        q, k, v = self.qkv_proj(h).chunk(3, dim=-1)

        # Reshape with -1 on the K axis so ONNX never bakes K as a constant
        q = q.reshape(1, -1, self.nhead, self.head_dim).transpose(1, 2)  # (1, nhead, K, head_dim)
        k = k.reshape(1, -1, self.nhead, self.head_dim).transpose(1, 2)  # (1, nhead, K, head_dim)
        v = v.reshape(1, -1, self.nhead, self.head_dim).transpose(1, 2)  # (1, nhead, K, head_dim)

        attn_scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale  # (1, nhead, K, K)
        attn_probs = self.attn_drop(F.softmax(attn_scores, dim=-1))
        context = torch.matmul(attn_probs, v)                            # (1, nhead, K, head_dim)
        context = context.transpose(1, 2).reshape(1, -1, self.d_model)   # (1, K, d_model)

        x = x + self.out_drop(self.out_proj(context))
        x = x + self.ffn(self.norm2(x))
        return x


class Etapa3CabezaSetDecisionAbstencion300M(nn.Module):
    """
    Stage 3: Deep Set-Attention Reranker + Cardinality-Aware (log K) Abstention Head.
    Jointly compares all K candidate tools via self-attention before producing calibrated scores
    and the global abstention probability.
    """
    def __init__(self, d_in: int = 512, nhead: int = 8, num_layers: int = 3, dropout: float = 0.1):
        super().__init__()
        self.set_layers = nn.ModuleList(
            [PreLNSetAttentionLayer(d_model=d_in, nhead=nhead, dropout=dropout) for _ in range(num_layers)]
        )
        self.score_head = nn.Sequential(
            nn.Linear(d_in, d_in // 2),
            nn.LayerNorm(d_in // 2),
            nn.GELU(),
            nn.Linear(d_in // 2, 1),
        )

        # 14 statistical + NLI features + 2*d_in (max_pool + mean_pool of set_context)
        in_abstencion = 14 + d_in * 2
        self.abstencion_mlp = nn.Sequential(
            nn.Linear(in_abstencion, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Linear(128, 1),
        )

    def forward(
        self,
        tool_reprs: torch.Tensor,
        estado_probs: torch.Tensor,
        gate_alpha: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        # 1. Contextualize candidate tools against each other (Set-Attention)
        x = tool_reprs.unsqueeze(0)                    # (1, K, d_in)
        for layer in self.set_layers:
            x = layer(x)
        set_context = x.squeeze(0)                     # (K, d_in)

        # 2. Raw logits & calibrated scores in (0, 1)
        raw_logits = self.score_head(set_context).squeeze(-1)  # (K,)
        ranked_scores = torch.sigmoid(raw_logits)              # (K,)

        # 3. Dynamic-shape-safe statistics over K candidates (works for any K >= 1 in ONNX without TracerWarnings)
        sorted_scores, _ = torch.sort(ranked_scores, descending=True)

        # Safe zero-padding to guarantee at least 3 elements without Python branching in ONNX trace
        padded_s = torch.cat([sorted_scores, torch.zeros(3, device=sorted_scores.device, dtype=sorted_scores.dtype)], dim=0)
        top1 = padded_s[0:1]
        top2 = padded_s[1:2]
        top3 = padded_s[2:3]
        margen_12 = top1 - top2
        margen_23 = top2 - top3

        # Normalized Shannon entropy in [0, 1] independent of K (k_tensor computed dynamically via ReduceSum)
        probs_dist = F.softmax(raw_logits, dim=-1)
        raw_entropy = -torch.sum(probs_dist * torch.log(probs_dist + 1e-8), dim=-1, keepdim=True)
        k_tensor = torch.ones_like(raw_logits).sum(dim=-1, keepdim=True)
        max_entropy = torch.log(k_tensor + 1.0).clamp(min=1.0)
        norm_entropy = raw_entropy / max_entropy

        # Explicit normalized cardinality signal: log(K + 1) / log(201)
        log_k_norm = torch.log(k_tensor + 1.0) / 5.3033049

        mean_s = torch.mean(ranked_scores, dim=-1, keepdim=True)
        std_s = torch.std(ranked_scores, unbiased=False, dim=-1, keepdim=True)

        max_pool, _ = torch.max(set_context, dim=0, keepdim=True)  # (1, d_in)
        mean_pool = torch.mean(set_context, dim=0, keepdim=True)   # (1, d_in)

        stats_vec = torch.cat(
            [
                top1.unsqueeze(0),
                top2.unsqueeze(0),
                top3.unsqueeze(0),
                margen_12.unsqueeze(0),
                margen_23.unsqueeze(0),
                norm_entropy.unsqueeze(0),
                mean_s.unsqueeze(0),
                std_s.unsqueeze(0),
                log_k_norm.unsqueeze(0),
                estado_probs,  # (1, 4)
                gate_alpha,    # (1, 1)
                max_pool,      # (1, d_in)
                mean_pool,     # (1, d_in)
            ],
            dim=-1,
        )  # (1, 14 + 2*d_in)

        abstencion_prob = torch.sigmoid(self.abstencion_mlp(stats_vec))  # (1, 1)
        return ranked_scores, abstencion_prob, raw_logits


class RemtkJuicioMoE(nn.Module):
    """
    Unified RemtkJuicioMoE Engine (supports both '300M' and '48M' configurations).
    Maintains the exact 3-input, 5-output ONNX contract required by the Rust/TypeScript runtime.
    """
    def __init__(
        self,
        d_model: int = 384,
        d_proj: int = 1024,
        d_out: int = 512,
        num_experts: int = 6,
        top_k: int = 2,
        nhead: int = 8,
        set_layers: int = 3,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.d_model = d_model
        self.d_proj = d_proj
        self.d_out = d_out
        self.num_experts = num_experts
        self.top_k = top_k

        self.etapa1 = Etapa1CompresorEstadoNLI(d_model=d_model)
        self.etapa2 = Etapa2MixtureOfExperts300M(
            d_model=d_model,
            d_proj=d_proj,
            d_out=d_out,
            num_experts=num_experts,
            top_k=top_k,
            dropout=dropout,
        )
        self.etapa3 = Etapa3CabezaSetDecisionAbstencion300M(
            d_in=d_out,
            nhead=nhead,
            num_layers=set_layers,
            dropout=dropout,
        )

    @staticmethod
    def _compute_valid_mask(tokens: torch.Tensor) -> torch.Tensor:
        """
        Automatic in-graph padding detection via L2 norm:
        Valid tokens from e5-small have ||v||_2 > 1e-5; zero-padded positions have ||v||_2 == 0.
        Always forces index 0 to be valid (1.0) to prevent NaN on empty sequences.
        """
        mask = (torch.norm(tokens, p=2, dim=-1) > 1e-5).float()
        first_col = torch.ones_like(mask[:, :1])
        return torch.cat([first_col, mask[:, 1:]], dim=1)

    def forward(
        self,
        query_tokens: torch.Tensor,
        context_tokens: torch.Tensor,
        tools_tokens: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        q_mask = self._compute_valid_mask(query_tokens)      # (1, q_len)
        c_mask = self._compute_valid_mask(context_tokens)    # (1, ctx_len)
        t_mask = self._compute_valid_mask(tools_tokens)      # (K, tool_len)

        q_enriched_seq, q_enriched_pool, c_pool, estado_probs, topic_shift_score, gate_alpha = self.etapa1(
            query_tokens, context_tokens, q_mask, c_mask
        )
        tool_reprs, sparse_gates, _ = self.etapa2(
            q_enriched_seq, q_enriched_pool, c_pool, estado_probs, tools_tokens, q_mask, t_mask
        )
        ranked_scores, abstencion_prob, _ = self.etapa3(tool_reprs, estado_probs, gate_alpha)

        return ranked_scores, estado_probs, abstencion_prob, topic_shift_score, sparse_gates

    def forward_train(
        self,
        query_tokens: torch.Tensor,
        context_tokens: torch.Tensor,
        tools_tokens: torch.Tensor,
    ) -> Dict[str, torch.Tensor]:
        """Extended forward pass returning intermediate logits & gates for multi-task loss calculation."""
        q_mask = self._compute_valid_mask(query_tokens)
        c_mask = self._compute_valid_mask(context_tokens)
        t_mask = self._compute_valid_mask(tools_tokens)

        q_enriched_seq, q_enriched_pool, c_pool, estado_probs, topic_shift_score, gate_alpha = self.etapa1(
            query_tokens, context_tokens, q_mask, c_mask
        )
        tool_reprs, sparse_gates, router_logits = self.etapa2(
            q_enriched_seq, q_enriched_pool, c_pool, estado_probs, tools_tokens, q_mask, t_mask
        )
        ranked_scores, abstencion_prob, raw_score_logits = self.etapa3(tool_reprs, estado_probs, gate_alpha)

        return {
            "ranked_scores": ranked_scores,
            "raw_score_logits": raw_score_logits,
            "estado_probs": estado_probs,
            "abstencion_prob": abstencion_prob,
            "topic_shift_score": topic_shift_score,
            "gate_alpha": gate_alpha,
            "sparse_gates": sparse_gates,
            "router_logits": router_logits,
        }


# Verify parameter count for both profiles (300M and 48M) and instantiate the selected profile
for prof_name, cfg in MODEL_PROFILES.items():
    tmp_model = RemtkJuicioMoE(**cfg)
    n_params = sum(p.numel() for p in tmp_model.parameters())
    fp32_mb = (n_params * 4) / (1024 ** 2)
    int8_mb = (n_params * 1) / (1024 ** 2)
    print(f"[Architecture Profile '{prof_name}'] Total Parameters: {n_params:,} ({n_params/1e6:.2f}M) | FP32: {fp32_mb:.1f} MB | INT8: ~{int8_mb:.1f} MB")
    del tmp_model

ACTIVE_CFG = MODEL_PROFILES[ACTIVE_PROFILE]
model = RemtkJuicioMoE(**ACTIVE_CFG).to(DEVICE)
print(f"\n[Active Model] Initialized RemtkJuicioMoE ({ACTIVE_PROFILE}) on {DEVICE}")

---

## Section 4: Tokenizer & Representation Precomputation (`intfloat/multilingual-e5-small`)
Precomputes token-level `last_hidden_state` sequences (`d_model = 384`) using `intfloat/multilingual-e5-small`:
- Applies the mandatory E5 prefixes (`"query: "` for user queries and `"passage: "` for conversational context and tool descriptions).
- L2-normalizes valid token embeddings (`F.normalize(..., p=2, dim=-1)`) and **multiplies by `attention_mask.unsqueeze(-1)`** so padded positions are strictly `0.0` (enabling `RemtkJuicioMoE._compute_valid_mask` to detect padding automatically via L2 norm `> 1e-5`).
- Caches all catalog tool tensors (`(num_catalog_tools, MAX_TOOL_LEN, 384)`) and episode query/context tensors in RAM/disk (`cache_tensors/`) so epochs train in seconds.

In [ ]:
# ==============================================================================
# CELL 4: TOKENIZER & REPRESENTATION PRECOMPUTATION (multilingual-e5-small)
# ==============================================================================
from transformers import AutoTokenizer, AutoModel

BACKBONE_NAME = "intfloat/multilingual-e5-small"
MAX_QUERY_LEN = 36
MAX_CTX_LEN = 48
MAX_TOOL_LEN = 32
BATCH_ENCODE_SIZE = 64

print(f"[Backbone] Loading tokenizer & encoder '{BACKBONE_NAME}' on {DEVICE}...")
e5_tokenizer = AutoTokenizer.from_pretrained(BACKBONE_NAME)
e5_encoder = AutoModel.from_pretrained(BACKBONE_NAME).to(DEVICE)
e5_encoder.eval()

@torch.no_grad()
def encode_token_sequences(
    texts: List[str],
    prefix: str,
    max_len: int,
    batch_size: int = BATCH_ENCODE_SIZE,
) -> torch.Tensor:
    """
    Encodes a list of strings into L2-normalized token-level embeddings of shape (N, max_len, 384).
    Crucially masks padded positions to exact 0.0 so the ONNX in-graph L2-norm mask detects them cleanly.
    """
    prefixed_texts = [f"{prefix}{t.strip()}" if not t.strip().startswith(prefix) else t.strip() for t in texts]
    outputs_list = []

    for start_idx in range(0, len(prefixed_texts), batch_size):
        batch_slice = prefixed_texts[start_idx : start_idx + batch_size]
        encoded = e5_tokenizer(
            batch_slice,
            padding="max_length",
            truncation=True,
            max_length=max_len,
            return_tensors="pt",
        ).to(DEVICE)

        out = e5_encoder(**encoded)
        hidden = F.normalize(out.last_hidden_state, p=2, dim=-1)  # (B, L, 384)
        attn_mask = encoded["attention_mask"].unsqueeze(-1).float()  # (B, L, 1)

        # Zero-out padding tokens strictly so ||pad_token||_2 == 0.0
        masked_hidden = (hidden * attn_mask).cpu().to(torch.float16)
        outputs_list.append(masked_hidden)

    return torch.cat(outputs_list, dim=0)

# 1. Precompute Catalog Tool Token Tensors
tool_id_to_idx = {tid: idx for idx, tid in enumerate(ALL_TOOL_IDS)}
tool_texts = [CATALOG[tid]["text_for_embedding"] for tid in ALL_TOOL_IDS]

t0 = time.perf_counter()
print(f"[Precompute] Encoding {len(ALL_TOOL_IDS)} catalog tools (max_len={MAX_TOOL_LEN})...")
CATALOG_TOOL_TENSORS = encode_token_sequences(tool_texts, prefix="passage: ", max_len=MAX_TOOL_LEN)
print(f"[Precompute] Catalog tensor shape: {tuple(CATALOG_TOOL_TENSORS.shape)} ({CATALOG_TOOL_TENSORS.element_size() * CATALOG_TOOL_TENSORS.nelement() / (1024**2):.2f} MB)")

# 2. Precompute Episode Query & Context Token Tensors
query_texts = [s["query"] for s in DATASET_SAMPLES]
context_texts = [s["context"] for s in DATASET_SAMPLES]

print(f"[Precompute] Encoding {len(DATASET_SAMPLES)} episode queries (max_len={MAX_QUERY_LEN}) and contexts (max_len={MAX_CTX_LEN})...")
QUERY_TENSORS = encode_token_sequences(query_texts, prefix="query: ", max_len=MAX_QUERY_LEN)
CONTEXT_TENSORS = encode_token_sequences(context_texts, prefix="passage: ", max_len=MAX_CTX_LEN)
elapsed_pre = time.perf_counter() - t0

# Verify padding mask detection on sample 0
sample_q_fp32 = QUERY_TENSORS[0:1].to(torch.float32)
detected_mask = RemtkJuicioMoE._compute_valid_mask(sample_q_fp32)
valid_tok_count = int(detected_mask.sum().item())
print(f"[Precompute] Completed in {elapsed_pre:.2f}s | Sample[0] valid query tokens: {valid_tok_count}/{MAX_QUERY_LEN}")

# Free e5_encoder from GPU VRAM during training if needed (kept on CPU for later live inference tests)
e5_encoder = e5_encoder.cpu()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

---

## Section 5: Multi-Task Loss ($\mathcal{L}_{\text{Total}}$) & Training Loop
Implements the unified 6-component loss from `docs/plan/plan_03_entrenamiento_en_cpu_y_funci_n_de_p_rdida.md`:
$$\mathcal{L}_{\text{Total}} = \lambda_1 \mathcal{L}_{\text{ListNet/BCE}} + \lambda_2 \mathcal{L}_{\text{Margin}} + \lambda_3 \mathcal{L}_{\text{EstadoNLI}} + \lambda_4 \mathcal{L}_{\text{Abstención (Focal)}} + \lambda_5 \mathcal{L}_{\text{BalanceMoE + Hint}} + \lambda_6 \mathcal{L}_{\text{ContrastiveGate}}$$
- **$\mathcal{L}_{\text{ListNet/BCE}}$ ($\lambda_1 = 1.0$)**: Combines temperature-scaled ListNet KL divergence over candidate distributions with pointwise binary cross-entropy calibration.
- **$\mathcal{L}_{\text{Margin}}$ ($\lambda_2 = 0.8$)**: Enforces a minimum separation margin ($\ge 0.35$) between the positive tool and the hardest negative candidate.
- **$\mathcal{L}_{\text{EstadoNLI}}$ ($\lambda_3 = 0.6$)**: Soft-target KL divergence on the 4 conversational states (`[TRANSICION, CONTINUACION, CORRECCION, ABANDONO]`).
- **$\mathcal{L}_{\text{Abstención}}$ ($\lambda_4 = 1.2$)**: Binary Focal Loss ($\gamma = 2.0$) penalizing false tool executions when the user is making small talk or asking out-of-domain questions.
- **$\mathcal{L}_{\text{BalanceMoE + Hint}}$ ($\lambda_5 = 0.25$)**: Switch-Transformer auxiliary load-balancing penalty $N \sum_i f_i P_i$ plus domain-expert routing supervision.
- **$\mathcal{L}_{\text{ContrastiveGate}}$ ($\lambda_6 = 0.4$)**: Supervises `gate_alpha` so the model blocks stale context during `TRANSICION`/`ABANDONO` ($\alpha \to 1$) and retains context during `CONTINUACION` ($\alpha \to 0$).
- Includes **Token Dropout ($p=0.10$)** on valid tokens during training, gradient accumulation, `AdamW`, `OneCycleLR`, gradient clipping (`max_norm=1.0`), checkpointing (`best_moe.pt`), and training loss curve visualization.

In [ ]:
# ==============================================================================
# CELL 5: MULTI-TASK LOSS & TRAINING LOOP
# ==============================================================================

class RemtkMoEMultiTaskLoss(nn.Module):
    """
    6-Component Multi-Task Loss for RemtkJuicioMoE:
      1. ListNet KL + Pointwise BCE Calibration (Ranking Distillation)
      2. Pairwise Hard-Negative Margin Ranking Loss
      3. NLI Conversational State Soft-KL Loss (4 states)
      4. Focal Loss for Cardinality-Aware Abstention
      5. MoE Switch Load-Balancing + Functional Domain Hint Loss
      6. Contrastive Context Gate (gate_alpha) Supervision Loss
    """
    def __init__(
        self,
        num_experts: int = 6,
        temperature: float = 1.5,
        margin: float = 0.35,
        lambda_rank: float = 1.0,
        lambda_margin: float = 0.8,
        lambda_nli: float = 0.6,
        lambda_abst: float = 1.2,
        lambda_moe: float = 0.25,
        lambda_gate: float = 0.4,
    ):
        super().__init__()
        self.num_experts = num_experts
        self.temperature = temperature
        self.margin = margin
        self.lambda_rank = lambda_rank
        self.lambda_margin = lambda_margin
        self.lambda_nli = lambda_nli
        self.lambda_abst = lambda_abst
        self.lambda_moe = lambda_moe
        self.lambda_gate = lambda_gate

    def forward(
        self,
        out: Dict[str, torch.Tensor],
        teacher_scores: torch.Tensor,
        estado_target: torch.Tensor,
        abstencion_target: torch.Tensor,
        gate_target: torch.Tensor,
        expert_hint: int,
    ) -> Tuple[torch.Tensor, Dict[str, float]]:
        ranked_scores = out["ranked_scores"]          # (K,)
        raw_logits = out["raw_score_logits"]          # (K,)
        estado_probs = out["estado_probs"]            # (1, 4)
        abstencion_prob = out["abstencion_prob"]      # (1, 1)
        gate_alpha = out["gate_alpha"]                # (1, 1)
        router_logits = out["router_logits"]          # (1, num_experts)

        # 1. Ranking Loss: ListNet Softmax KL (only when not abstaining) + Pointwise BCE
        bce_rank = F.binary_cross_entropy(ranked_scores.clamp(1e-6, 1.0 - 1e-6), teacher_scores)
        if abstencion_target.item() < 0.5 and ranked_scores.numel() > 1:
            student_log_dist = F.log_softmax(raw_logits / self.temperature, dim=-1)
            teacher_dist = F.softmax((teacher_scores * 6.0) / self.temperature, dim=-1)
            listnet_kl = F.kl_div(student_log_dist, teacher_dist, reduction="batchmean") * (self.temperature ** 2)
            loss_rank = 0.6 * bce_rank + 0.4 * listnet_kl
        else:
            loss_rank = bce_rank

        # 2. Hard-Negative Pairwise Margin Loss: max(0, margin - (s_pos - s_hard_neg))
        if abstencion_target.item() < 0.5 and ranked_scores.numel() > 1:
            pos_idx = int(torch.argmax(teacher_scores).item())
            pos_score = ranked_scores[pos_idx]
            neg_mask = torch.ones_like(ranked_scores, dtype=torch.bool)
            neg_mask[pos_idx] = False
            hardest_neg_score = ranked_scores[neg_mask].max()
            loss_margin = F.relu(self.margin - (pos_score - hardest_neg_score))
        else:
            # During abstention, penalize any tool score exceeding 0.25
            loss_margin = F.relu(ranked_scores.max() - 0.25)

        # 3. NLI State Classification Loss (KL divergence against soft teacher distribution)
        log_estado = torch.log(estado_probs.clamp(min=1e-7))
        loss_nli = F.kl_div(log_estado, estado_target.unsqueeze(0), reduction="batchmean")

        # 4. Focal Abstention Loss (gamma = 2.0)
        p_abs = abstencion_prob.view(-1).clamp(1e-6, 1.0 - 1e-6)
        y_abs = abstencion_target.view(-1)
        pt = torch.where(y_abs >= 0.5, p_abs, 1.0 - p_abs)
        focal_weight = (1.0 - pt) ** 2.0
        bce_abs = F.binary_cross_entropy(p_abs, y_abs, reduction="none")
        loss_abst = (focal_weight * bce_abs).mean()

        # 5. MoE Routing Loss: Domain Expert Hint + Entropy Regularization (prevents expert collapse)
        target_expert = torch.tensor([expert_hint % self.num_experts], dtype=torch.long, device=router_logits.device)
        loss_expert_hint = F.cross_entropy(router_logits, target_expert)
        router_probs = F.softmax(router_logits, dim=-1)
        # Encourage high entropy across batch/episodes + smooth probability floor
        ideal_uniform = torch.full_like(router_probs, 1.0 / self.num_experts)
        loss_balance = F.mse_loss(router_probs, ideal_uniform) * self.num_experts
        loss_moe = 0.7 * loss_expert_hint + 0.3 * loss_balance

        # 6. Contrastive Context Gate Loss
        loss_gate = F.smooth_l1_loss(gate_alpha.view(-1), gate_target.view(-1))

        total_loss = (
            self.lambda_rank * loss_rank
            + self.lambda_margin * loss_margin
            + self.lambda_nli * loss_nli
            + self.lambda_abst * loss_abst
            + self.lambda_moe * loss_moe
            + self.lambda_gate * loss_gate
        )

        metrics = {
            "total": float(total_loss.item()),
            "rank": float(loss_rank.item()),
            "margin": float(loss_margin.item()),
            "nli": float(loss_nli.item()),
            "abst": float(loss_abst.item()),
            "moe": float(loss_moe.item()),
            "gate": float(loss_gate.item()),
        }
        return total_loss, metrics


def apply_token_dropout(tokens: torch.Tensor, drop_prob: float = 0.08) -> torch.Tensor:
    """
    Randomly zeros out a small fraction of non-padded tokens (keeping token 0 intact)
    to improve robustness against noisy queries and prevent over-reliance on single keywords.
    """
    if drop_prob <= 0.0 or tokens.size(1) <= 2:
        return tokens
    valid_mask = (torch.norm(tokens, p=2, dim=-1) > 1e-5)
    keep_mask = (torch.rand(tokens.shape[:2], device=tokens.device) > drop_prob).float()
    keep_mask[:, 0] = 1.0  # Never drop the first valid token
    # Ensure at least one valid token remains
    combined = (keep_mask * valid_mask.float()).unsqueeze(-1)
    return tokens * combined


# Train / Validation Split (85% Train, 15% Validation stratified across NLI states)
indices = list(range(len(DATASET_SAMPLES)))
random.Random(SEED).shuffle(indices)
split_idx = int(len(indices) * 0.85)
train_indices = indices[:split_idx]
val_indices = indices[split_idx:]

print(f"[Split] Training episodes: {len(train_indices)} | Validation episodes: {len(val_indices)}")

# Training Hyperparameters
EPOCHS = 8 if torch.cuda.is_available() else 5
GRAD_ACCUM_STEPS = 8
LEARNING_RATE = 3e-4 if ACTIVE_PROFILE == "48M" else 2e-4
WEIGHT_DECAY = 1e-2

criterion = RemtkMoEMultiTaskLoss(num_experts=ACTIVE_CFG["num_experts"]).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
total_optim_steps = max(1, (len(train_indices) * EPOCHS) // GRAD_ACCUM_STEPS)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=LEARNING_RATE,
    total_steps=total_optim_steps + EPOCHS,
    pct_start=0.15,
    anneal_strategy="cos",
)

history = {
    "train_loss": [],
    "val_loss": [],
    "val_top1": [],
    "val_nli_acc": [],
    "val_abst_acc": [],
}

best_val_score = -1.0
best_ckpt_path = CHECKPOINT_DIR / "best_moe.pt"

@torch.no_grad()
def evaluate_validation(indices_eval: List[int]) -> Dict[str, Any]:
    model.eval()
    losses = []
    top1_hits = 0
    top3_hits = 0
    rank_count = 0
    nli_preds, nli_targets = [], []
    abst_preds, abst_targets = [], []
    expert_loads = np.zeros(ACTIVE_CFG["num_experts"], dtype=np.float64)

    for idx in indices_eval:
        sample = DATASET_SAMPLES[idx]
        q_tok = QUERY_TENSORS[idx : idx + 1].to(DEVICE, dtype=torch.float32)
        c_tok = CONTEXT_TENSORS[idx : idx + 1].to(DEVICE, dtype=torch.float32)
        cand_indices = [tool_id_to_idx[cid] for cid in sample["candidate_ids"]]
        t_tok = CATALOG_TOOL_TENSORS[cand_indices].to(DEVICE, dtype=torch.float32)

        t_scores = torch.tensor(sample["teacher_scores"], dtype=torch.float32, device=DEVICE)
        est_target = torch.tensor(sample["estado_probs"], dtype=torch.float32, device=DEVICE)
        abs_target = torch.tensor([sample["abstencion_label"]], dtype=torch.float32, device=DEVICE)
        gate_target = torch.tensor([sample["gate_target"]], dtype=torch.float32, device=DEVICE)

        out = model.forward_train(q_tok, c_tok, t_tok)
        loss, _ = criterion(out, t_scores, est_target, abs_target, gate_target, sample["expert_hint"])
        losses.append(float(loss.item()))

        # Track ranking accuracy on non-abstention samples
        if sample["abstencion_label"] < 0.5:
            rank_count += 1
            gt_best = int(torch.argmax(t_scores).item())
            pred_order = torch.argsort(out["ranked_scores"], descending=True).cpu().tolist()
            if pred_order[0] == gt_best:
                top1_hits += 1
            if gt_best in pred_order[:3]:
                top3_hits += 1

        # Track NLI state & Abstention accuracy
        nli_preds.append(int(torch.argmax(out["estado_probs"], dim=-1).item()))
        nli_targets.append(int(sample["estado_label"]))

        abst_preds.append(float(out["abstencion_prob"].item()))
        abst_targets.append(int(sample["abstencion_label"] >= 0.5))

        expert_loads += out["sparse_gates"].squeeze(0).cpu().numpy()

    expert_loads = expert_loads / max(1, len(indices_eval))
    nli_acc = float(np.mean(np.array(nli_preds) == np.array(nli_targets)))
    abst_bin_preds = (np.array(abst_preds) >= 0.5).astype(int)
    abst_acc = float(np.mean(abst_bin_preds == np.array(abst_targets)))

    return {
        "loss": float(np.mean(losses)),
        "top1_acc": top1_hits / max(1, rank_count),
        "top3_acc": top3_hits / max(1, rank_count),
        "nli_acc": nli_acc,
        "abst_acc": abst_acc,
        "nli_preds": nli_preds,
        "nli_targets": nli_targets,
        "abst_preds": abst_preds,
        "abst_targets": abst_targets,
        "expert_loads": expert_loads,
    }


print("=" * 85)
print(f"STARTING MULTI-TASK TRAINING | Profile: {ACTIVE_PROFILE} | Epochs: {EPOCHS} | Device: {DEVICE}")
print("=" * 85)

train_start = time.perf_counter()
for epoch in range(1, EPOCHS + 1):
    model.train()
    random.shuffle(train_indices)
    epoch_losses = []
    comp_accum = {"rank": 0.0, "margin": 0.0, "nli": 0.0, "abst": 0.0, "moe": 0.0, "gate": 0.0}
    optimizer.zero_grad()

    pbar = tqdm(train_indices, desc=f"Epoch {epoch:02d}/{EPOCHS:02d}", leave=False)
    for step_idx, idx in enumerate(pbar, start=1):
        sample = DATASET_SAMPLES[idx]
        q_tok = QUERY_TENSORS[idx : idx + 1].to(DEVICE, dtype=torch.float32)
        c_tok = CONTEXT_TENSORS[idx : idx + 1].to(DEVICE, dtype=torch.float32)
        cand_indices = [tool_id_to_idx[cid] for cid in sample["candidate_ids"]]
        t_tok = CATALOG_TOOL_TENSORS[cand_indices].to(DEVICE, dtype=torch.float32)

        # Apply token dropout regularization during training
        q_tok = apply_token_dropout(q_tok, drop_prob=0.08)
        c_tok = apply_token_dropout(c_tok, drop_prob=0.08)

        t_scores = torch.tensor(sample["teacher_scores"], dtype=torch.float32, device=DEVICE)
        est_target = torch.tensor(sample["estado_probs"], dtype=torch.float32, device=DEVICE)
        abs_target = torch.tensor([sample["abstencion_label"]], dtype=torch.float32, device=DEVICE)
        gate_target = torch.tensor([sample["gate_target"]], dtype=torch.float32, device=DEVICE)

        out = model.forward_train(q_tok, c_tok, t_tok)
        loss, comp = criterion(out, t_scores, est_target, abs_target, gate_target, sample["expert_hint"])

        (loss / GRAD_ACCUM_STEPS).backward()
        epoch_losses.append(comp["total"])
        for k in comp_accum:
            comp_accum[k] += comp[k]

        if step_idx % GRAD_ACCUM_STEPS == 0 or step_idx == len(train_indices):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

    val_metrics = evaluate_validation(val_indices)
    mean_train_loss = float(np.mean(epoch_losses))

    history["train_loss"].append(mean_train_loss)
    history["val_loss"].append(val_metrics["loss"])
    history["val_top1"].append(val_metrics["top1_acc"])
    history["val_nli_acc"].append(val_metrics["nli_acc"])
    history["val_abst_acc"].append(val_metrics["abst_acc"])

    composite_score = 0.45 * val_metrics["top1_acc"] + 0.30 * val_metrics["nli_acc"] + 0.25 * val_metrics["abst_acc"]
    is_best = composite_score >= best_val_score
    if is_best:
        best_val_score = composite_score
        torch.save(
            {
                "epoch": epoch,
                "profile": ACTIVE_PROFILE,
                "config": ACTIVE_CFG,
                "model_state_dict": model.state_dict(),
                "val_metrics": {k: v for k, v in val_metrics.items() if isinstance(v, (int, float))},
            },
            best_ckpt_path,
        )

    star = "★ BEST" if is_best else ""
    print(
        f"Epoch {epoch:02d}/{EPOCHS:02d} | Train Loss: {mean_train_loss:.4f} | Val Loss: {val_metrics['loss']:.4f} | "
        f"Top-1: {val_metrics['top1_acc']:.1%} (Top-3: {val_metrics['top3_acc']:.1%}) | "
        f"NLI Acc: {val_metrics['nli_acc']:.1%} | Abst Acc: {val_metrics['abst_acc']:.1%} {star}"
    )

print(f"\n[Training Complete] Total time: {time.perf_counter() - train_start:.1f}s | Best checkpoint saved to: {best_ckpt_path}")

# Reload best checkpoint weights
ckpt = torch.load(best_ckpt_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

# Plot live training & validation curves
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
epochs_range = range(1, EPOCHS + 1)

axes[0].plot(epochs_range, history["train_loss"], marker="o", linewidth=2, label="Train Multi-Task Loss")
axes[0].plot(epochs_range, history["val_loss"], marker="s", linewidth=2, label="Val Multi-Task Loss")
axes[0].set_title(f"RemtkJuicioMoE ({ACTIVE_PROFILE}) — Multi-Task Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(epochs_range, [x * 100 for x in history["val_top1"]], marker="o", linewidth=2, label="Top-1 Tool Recall (%)")
axes[1].plot(epochs_range, [x * 100 for x in history["val_nli_acc"]], marker="^", linewidth=2, label="NLI State Acc (%)")
axes[1].plot(epochs_range, [x * 100 for x in history["val_abst_acc"]], marker="d", linewidth=2, label="Abstention Acc (%)")
axes[1].set_title("Validation Multi-Task Accuracy Metrics")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy (%)")
axes[1].set_ylim(0, 105)
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

---

## Section 6: Comprehensive Evaluation & Diagnostics
Evaluates the trained `RemtkJuicioMoE` checkpoint across four diagnostic dimensions:
1. **Ranking Quality**: Top-1 Accuracy, Top-3 Recall, and Mean Reciprocal Rank (MRR).
2. **NLI Conversational State Confusion Matrix**: `TRANSICION`, `CONTINUACION`, `CORRECCION` (distant negation), and `ABANDONO`.
3. **Abstention Calibration Curves**: ROC-AUC and Precision-Recall (PR-AUC) for out-of-domain / small-talk rejection.
4. **Sparse MoE Expert Load Distribution**: Verifies that Top-2 gating distributes traffic across all functional macro-experts without expert collapse.

In [ ]:
# ==============================================================================
# CELL 6: EVALUATION & DIAGNOSTICS (CONFUSION MATRIX, ROC/PR, EXPERT LOAD)
# ==============================================================================
from sklearn.metrics import (  # type: ignore[import-not-found]  # type: ignore[import-not-found]
    confusion_matrix,
    classification_report,
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score,
)

final_eval = evaluate_validation(val_indices)

# Compute Mean Reciprocal Rank (MRR) on validation set
reciprocal_ranks = []
for idx in val_indices:
    sample = DATASET_SAMPLES[idx]
    if sample["abstencion_label"] >= 0.5:
        continue
    q_tok = QUERY_TENSORS[idx : idx + 1].to(DEVICE, dtype=torch.float32)
    c_tok = CONTEXT_TENSORS[idx : idx + 1].to(DEVICE, dtype=torch.float32)
    cand_indices = [tool_id_to_idx[cid] for cid in sample["candidate_ids"]]
    t_tok = CATALOG_TOOL_TENSORS[cand_indices].to(DEVICE, dtype=torch.float32)
    with torch.no_grad():
        ranked_scores, _, _, _, _ = model(q_tok, c_tok, t_tok)
    gt_best = int(np.argmax(sample["teacher_scores"]))
    sorted_idx = torch.argsort(ranked_scores, descending=True).cpu().tolist()
    rank_pos = sorted_idx.index(gt_best) + 1
    reciprocal_ranks.append(1.0 / rank_pos)

mrr_score = float(np.mean(reciprocal_ranks)) if reciprocal_ranks else 0.0

state_names = ["TRANSICION", "CONTINUACION", "CORRECCION", "ABANDONO"]
cm = confusion_matrix(final_eval["nli_targets"], final_eval["nli_preds"], labels=[0, 1, 2, 3])

# ROC & PR curves for Abstention
y_true_abs = np.array(final_eval["abst_targets"])
y_score_abs = np.array(final_eval["abst_preds"])
if len(np.unique(y_true_abs)) > 1:
    fpr, tpr, _ = roc_curve(y_true_abs, y_score_abs)
    roc_auc = auc(fpr, tpr)
    prec, rec, _ = precision_recall_curve(y_true_abs, y_score_abs)
    pr_auc = average_precision_score(y_true_abs, y_score_abs)
else:
    fpr, tpr, roc_auc = [0, 1], [0, 1], 1.0
    prec, rec, pr_auc = [1, 0], [0, 1], 1.0

print("=" * 80)
print(f"VALIDATION DIAGNOSTIC REPORT — RemtkJuicioMoE ({ACTIVE_PROFILE})")
print("=" * 80)
print(f"Tool Ranking Top-1 Accuracy : {final_eval['top1_acc']:.2%}")
print(f"Tool Ranking Top-3 Recall   : {final_eval['top3_acc']:.2%}")
print(f"Mean Reciprocal Rank (MRR)  : {mrr_score:.4f}")
print(f"NLI State Accuracy (4-way)  : {final_eval['nli_acc']:.2%}")
print(f"Abstention Accuracy         : {final_eval['abst_acc']:.2%} (ROC-AUC: {roc_auc:.4f} | PR-AUC: {pr_auc:.4f})")
print("-" * 80)
print("NLI Conversational State Classification Report:")
print(classification_report(final_eval["nli_targets"], final_eval["nli_preds"], target_names=state_names, digits=3, zero_division=0))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# 1. NLI Confusion Matrix
im = axes[0].imshow(cm, cmap="Blues")
axes[0].set_title("NLI Conversational State Confusion Matrix")
axes[0].set_xticks(range(4))
axes[0].set_yticks(range(4))
axes[0].set_xticklabels(state_names, rotation=30, ha="right")
axes[0].set_yticklabels(state_names)
axes[0].set_xlabel("Predicted State")
axes[0].set_ylabel("True State")
for r in range(4):
    for c in range(4):
        axes[0].text(c, r, str(cm[r, c]), ha="center", va="center", color="white" if cm[r, c] > cm.max() / 2 else "black", fontweight="bold")

# 2. Abstention ROC & PR Curves
axes[1].plot(fpr, tpr, label=f"ROC Curve (AUC = {roc_auc:.3f})", linewidth=2, color="#1f77b4")
axes[1].plot(rec, prec, label=f"PR Curve (AP = {pr_auc:.3f})", linewidth=2, linestyle="--", color="#ff7f0e")
axes[1].plot([0, 1], [0, 1], color="gray", alpha=0.4, linestyle=":")
axes[1].set_title("Cardinality-Aware Abstention ROC & PR Curves")
axes[1].set_xlabel("False Positive Rate / Recall")
axes[1].set_ylabel("True Positive Rate / Precision")
axes[1].legend(loc="lower left")
axes[1].grid(True, alpha=0.3)

# 3. Sparse MoE Expert Load Distribution
num_exp = ACTIVE_CFG["num_experts"]
exp_labels = [f"E{i}\n({EXPERT_DOMAINS.get(i, 'AUX')[:10]})" for i in range(num_exp)]
loads = final_eval["expert_loads"] * 100.0
bars = axes[2].bar(exp_labels, loads, color="#2ca02c", alpha=0.85, edgecolor="black")
axes[2].axhline(100.0 / num_exp, color="red", linestyle="--", linewidth=1.5, label=f"Ideal Uniform ({100.0/num_exp:.1f}%)")
axes[2].set_title("Top-2 Sparse MoE Routing Load Distribution")
axes[2].set_ylabel("Mean Routing Weight (%)")
axes[2].set_ylim(0, max(35.0, float(loads.max()) * 1.25))
axes[2].legend()
axes[2].grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

---

## Section 7: ONNX Export (`opset_version=17`, Dynamic Axes) & INT8 Dynamic Quantization
Exports `RemtkJuicioMoE` to the production ONNX format required by the Rust (`ort`) and TypeScript (`onnxruntime-node`) inference services:
- **Dynamic Axes**:
  - `query_tokens`: `{1: "q_len"}`
  - `context_tokens`: `{1: "ctx_len"}`
  - `tools_tokens`: `{0: "num_tools", 1: "tool_len"}`
  - `ranked_scores`: `{0: "num_tools"}`
- **Numerical Parity Check**: Verifies that the FP32 ONNX graph matches PyTorch outputs within $\le 10^{-4}$ across variable $K$ and sequence lengths.
- **INT8 Dynamic Quantization (`quantize_dynamic`)**: Applies `QuantType.QInt8` weight quantization (`MatMul` and `Gemm` operators) to produce `model_quantized.onnx` (reducing footprint by ~4x).

In [ ]:
# ==============================================================================
# CELL 7: ONNX EXPORT (OPSET 17) & INT8 DYNAMIC QUANTIZATION
# ==============================================================================
from onnxruntime.quantization import quantize_dynamic, QuantType

fp32_onnx_path = EXPORT_DIR / "model_fp32.onnx"
int8_onnx_path = EXPORT_DIR / "model_quantized.onnx"
metadata_json_path = EXPORT_DIR / "moe_metadata.json"

model_cpu = RemtkJuicioMoE(**ACTIVE_CFG).cpu()
model_cpu.load_state_dict(model.cpu().state_dict())
model_cpu.eval()

# Dummy inputs for tracing with padding tokens at the tail to verify mask tracing
dummy_q = F.normalize(torch.randn(1, 24, 384), p=2, dim=-1)
dummy_q[:, 18:, :] = 0.0  # Simulate 6 padded tokens

dummy_c = F.normalize(torch.randn(1, 32, 384), p=2, dim=-1)
dummy_c[:, 25:, :] = 0.0  # Simulate 7 padded tokens

dummy_t = F.normalize(torch.randn(16, 28, 384), p=2, dim=-1)
dummy_t[:, 20:, :] = 0.0  # Simulate 8 padded tokens

input_names = ["query_tokens", "context_tokens", "tools_tokens"]
output_names = ["ranked_scores", "estado_probs", "abstencion_prob", "topic_shift_score", "sparse_gates"]
dynamic_axes = {
    "query_tokens": {1: "q_len"},
    "context_tokens": {1: "ctx_len"},
    "tools_tokens": {0: "num_tools", 1: "tool_len"},
    "ranked_scores": {0: "num_tools"},
}

print(f"[ONNX Export] Exporting FP32 graph (opset_version=17) to {fp32_onnx_path}...")
with torch.no_grad():
    torch.onnx.export(
        model_cpu,
        (dummy_q, dummy_c, dummy_t),
        str(fp32_onnx_path),
        export_params=True,
        opset_version=17,
        do_constant_folding=True,
        input_names=input_names,
        output_names=output_names,
        dynamic_axes=dynamic_axes,
        dynamo=False,
    )

# Validate ONNX graph structure
onnx_model = onnx.load(str(fp32_onnx_path))
onnx.checker.check_model(onnx_model)
fp32_size_mb = fp32_onnx_path.stat().st_size / (1024 ** 2)
print(f"[ONNX Export] Graph check passed! FP32 size: {fp32_size_mb:.2f} MB")

# Numerical Parity Test (PyTorch vs ONNX FP32 on dynamic shapes: K=25, q_len=31, ctx_len=40, tool_len=26)
test_q = F.normalize(torch.randn(1, 31, 384), p=2, dim=-1)
test_q[:, 22:, :] = 0.0
test_c = F.normalize(torch.randn(1, 40, 384), p=2, dim=-1)
test_c[:, 30:, :] = 0.0
test_t = F.normalize(torch.randn(25, 26, 384), p=2, dim=-1)
test_t[:, 19:, :] = 0.0

with torch.no_grad():
    pt_outputs = model_cpu(test_q, test_c, test_t)

sess_fp32 = ort.InferenceSession(str(fp32_onnx_path), providers=["CPUExecutionProvider"])
ort_inputs = {
    "query_tokens": test_q.numpy(),
    "context_tokens": test_c.numpy(),
    "tools_tokens": test_t.numpy(),
}
ort_outputs: List[np.ndarray] = [np.asarray(x, dtype=np.float32) for x in sess_fp32.run(output_names, ort_inputs)]

print("\n[Parity Check] Comparing PyTorch FP32 vs ONNX Runtime FP32 across dynamic axes:")
max_diffs = {}
for name, pt_t, ort_arr in zip(output_names, pt_outputs, ort_outputs):
    diff = float(np.max(np.abs(pt_t.numpy() - ort_arr)))
    max_diffs[name] = diff
    status = "OK" if diff < 1e-4 else "WARN"
    print(f"  - {name:18s} | shape={str(ort_arr.shape):12s} | max_abs_diff = {diff:.2e} [{status}]")

# Apply INT8 Dynamic Quantization
print(f"\n[Quantization] Applying INT8 Dynamic Quantization -> {int8_onnx_path}...")
quantize_dynamic(
    model_input=str(fp32_onnx_path),
    model_output=str(int8_onnx_path),
    weight_type=QuantType.QInt8,
    op_types_to_quantize=["MatMul", "Gemm"],
    per_channel=True,
)
int8_size_mb = int8_onnx_path.stat().st_size / (1024 ** 2)
compression_ratio = fp32_size_mb / max(1e-6, int8_size_mb)
print(f"[Quantization] Complete! INT8 size: {int8_size_mb:.2f} MB (Compression ratio: {compression_ratio:.2f}x)")

# Save deployment metadata contract
metadata_payload = {
    "model_name": f"remtk-juicio-moe-{ACTIVE_PROFILE.lower()}-int8",
    "profile": ACTIVE_PROFILE,
    "architecture_config": ACTIVE_CFG,
    "backbone": BACKBONE_NAME,
    "opset_version": 17,
    "inputs": {
        "query_tokens": ["1", "q_len", 384],
        "context_tokens": ["1", "ctx_len", 384],
        "tools_tokens": ["num_tools", "tool_len", 384],
    },
    "outputs": {
        "ranked_scores": ["num_tools"],
        "estado_probs": [1, 4],
        "abstencion_prob": [1, 1],
        "topic_shift_score": [1, 1],
        "sparse_gates": [1, ACTIVE_CFG["num_experts"]],
    },
    "estado_labels": ["TRANSICION", "CONTINUACION", "CORRECCION", "ABANDONO"],
    "expert_domains": EXPERT_DOMAINS,
    "fp32_size_mb": round(fp32_size_mb, 2),
    "int8_size_mb": round(int8_size_mb, 2),
    "validation_metrics": {
        "top1_acc": round(final_eval["top1_acc"], 4),
        "top3_acc": round(final_eval["top3_acc"], 4),
        "mrr": round(mrr_score, 4),
        "nli_acc": round(final_eval["nli_acc"], 4),
        "abst_acc": round(final_eval["abst_acc"], 4),
    },
}
with open(metadata_json_path, "w", encoding="utf-8") as f:
    json.dump(metadata_payload, f, indent=2, ensure_ascii=False)
print(f"[Metadata] Saved deployment contract metadata to {metadata_json_path}")

---

## Section 8: Latency Benchmark, End-to-End Inference Tests & Artifact Packaging
1. **Latency Benchmark**: Measures P50, P95, and P99 latency (in milliseconds) of `model_quantized.onnx` across candidate set sizes $K \in \{15, 40, 80, 120\}$.
2. **End-to-End Qualitative Scenarios**:
   - **Scenario A (Direct Action / Continuity)**: User asks to schedule a recurring cron job (`cron_schedule_recurring_job`).
   - **Scenario B (Distant Negation / Correction)**: *"Espera, no vayas a borrar la cuenta del usuario... mejor actualiza su perfil"* (`users_update_profile` beats `users_delete_account`).
   - **Scenario C (Small-Talk / Out-of-Domain Abstention)**: *"Hola buenos días, cuéntame un chiste corto"* (`abstencion_prob > 0.5`).
3. **Artifact Bundle Packaging**: Packages `model_quantized.onnx`, `moe_metadata.json`, and `best_moe.pt` into `remtk-juicio-moe-int8-bundle.zip` ready for download or direct deployment into `remtk-prediction/models/remtk-juicio-moe-int8/`.

In [ ]:
# ==============================================================================
# CELL 8: LATENCY BENCHMARK, END-TO-END INFERENCE TESTS & ZIP PACKAGING
# ==============================================================================

sess_options = ort.SessionOptions()
sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
sess_options.intra_op_num_threads = min(8, max(2, (os.cpu_count() or 4) // 2))
sess_int8 = ort.InferenceSession(str(int8_onnx_path), sess_options=sess_options, providers=["CPUExecutionProvider"])

print("=" * 80)
print(f"ONNX RUNTIME INT8 CPU LATENCY BENCHMARK (Threads={sess_options.intra_op_num_threads})")
print("=" * 80)

k_benchmarks = [15, 40, 80, 120]
for k_val in k_benchmarks:
    b_q = F.normalize(torch.randn(1, 24, 384), p=2, dim=-1).numpy()
    b_c = F.normalize(torch.randn(1, 32, 384), p=2, dim=-1).numpy()
    b_t = F.normalize(torch.randn(k_val, 28, 384), p=2, dim=-1).numpy()
    feed = {"query_tokens": b_q, "context_tokens": b_c, "tools_tokens": b_t}

    # Warmup
    for _ in range(5):
        _ = sess_int8.run(output_names, feed)

    latencies_ms = []
    for _ in range(30):
        t_start = time.perf_counter()
        _ = sess_int8.run(output_names, feed)
        latencies_ms.append((time.perf_counter() - t_start) * 1000.0)

    p50 = float(np.percentile(latencies_ms, 50))
    p95 = float(np.percentile(latencies_ms, 95))
    p99 = float(np.percentile(latencies_ms, 99))
    print(f"Candidates K={k_val:3d} | P50: {p50:6.2f} ms | P95: {p95:6.2f} ms | P99: {p99:6.2f} ms")


# Live End-to-End Inference Helper using e5-small + INT8 ONNX Session
e5_encoder_cpu = e5_encoder.cpu().eval()

@torch.no_grad()
def run_live_int8_prediction(query: str, context: str, candidate_tool_ids: List[str]) -> Dict[str, Any]:
    q_enc = e5_tokenizer(f"query: {query}", padding="max_length", truncation=True, max_length=MAX_QUERY_LEN, return_tensors="pt")
    c_enc = e5_tokenizer(f"passage: {context}", padding="max_length", truncation=True, max_length=MAX_CTX_LEN, return_tensors="pt")

    q_hid = F.normalize(e5_encoder_cpu(**q_enc).last_hidden_state, p=2, dim=-1) * q_enc["attention_mask"].unsqueeze(-1).float()
    c_hid = F.normalize(e5_encoder_cpu(**c_enc).last_hidden_state, p=2, dim=-1) * c_enc["attention_mask"].unsqueeze(-1).float()

    cand_indices = [tool_id_to_idx[cid] for cid in candidate_tool_ids]
    t_hid = CATALOG_TOOL_TENSORS[cand_indices].to(torch.float32)

    t0 = time.perf_counter()
    raw_ort_out = sess_int8.run(
        output_names,
        {
            "query_tokens": q_hid.numpy().astype(np.float32),
            "context_tokens": c_hid.numpy().astype(np.float32),
            "tools_tokens": t_hid.numpy().astype(np.float32),
        },
    )
    ranked_scores = np.asarray(raw_ort_out[0], dtype=np.float32)
    estado_probs = np.asarray(raw_ort_out[1], dtype=np.float32)
    abstencion_prob = np.asarray(raw_ort_out[2], dtype=np.float32)
    topic_shift_score = np.asarray(raw_ort_out[3], dtype=np.float32)
    sparse_gates = np.asarray(raw_ort_out[4], dtype=np.float32)
    infer_ms = (time.perf_counter() - t0) * 1000.0

    order = np.argsort(-ranked_scores)
    top_tools = [(candidate_tool_ids[i], float(ranked_scores[i])) for i in order[:3]]
    est_idx = int(np.argmax(estado_probs[0]))
    active_experts = np.argsort(-sparse_gates[0])[:2]

    return {
        "top_tools": top_tools,
        "estado_pred": state_names[est_idx],
        "estado_probs": {state_names[i]: round(float(estado_probs[0, i]), 3) for i in range(4)},
        "abstencion_prob": float(abstencion_prob[0, 0]),
        "topic_shift_score": float(topic_shift_score[0, 0]),
        "top2_experts": [(int(e), EXPERT_DOMAINS.get(int(e), "AUX"), round(float(sparse_gates[0, e]), 3)) for e in active_experts],
        "latency_ms": infer_ms,
    }


print("\n" + "=" * 80)
print("LIVE END-TO-END INFERENCE VERIFICATION (e5-small + RemtkJuicioMoE INT8 ONNX)")
print("=" * 80)

sample_pool = ALL_TOOL_IDS[:24]
test_cases = [
    (
        "Scenario A — Direct Scheduling Action (Continuity)",
        "Necesito programar una tarea recurrente (cron job) para automatizar la rutina diaria.",
        "Usuario configurando tareas programadas en el módulo TEMPORAL_SCHEDULING.",
    ),
    (
        "Scenario B — Distant Negation & Correction",
        "Espera, no vayas a borrar usuario ni dar de baja la cuenta porque fue un error; mejor procede únicamente a modificar perfil y actualizar datos de usuario.",
        "El sistema propuso inicialmente ejecutar users_delete_account.",
    ),
    (
        "Scenario C — Out-of-Domain / Small-Talk Abstention",
        "Hola buenos días, cuéntame un chiste corto de programadores para relajar el ambiente.",
        "Última consulta: estado del servidor de producción.",
    ),
]

for title, q_str, c_str in test_cases:
    res = run_live_int8_prediction(q_str, c_str, sample_pool)
    decision = "ABSTAIN (No Tool)" if res["abstencion_prob"] >= 0.50 else f"EXECUTE -> {res['top_tools'][0][0]}"
    print(f"\n[{title}]")
    print(f"  Query           : {q_str}")
    print(f"  Decision        : {decision} (Abstention Prob: {res['abstencion_prob']:.3f} | Latency: {res['latency_ms']:.2f} ms)")
    print(f"  NLI State       : {res['estado_pred']} {res['estado_probs']} | Topic Shift: {res['topic_shift_score']:.3f}")
    print(f"  Active Experts  : {res['top2_experts']}")
    print(f"  Top-3 Candidates: {[(t, round(s, 4)) for t, s in res['top_tools']]}")

# Package artifacts into a downloadable ZIP archive
zip_bundle_path = WORKSPACE_DIR / "remtk-juicio-moe-int8-bundle.zip"
with zipfile.ZipFile(zip_bundle_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(int8_onnx_path, arcname="remtk-juicio-moe-int8/model_quantized.onnx")
    zf.write(metadata_json_path, arcname="remtk-juicio-moe-int8/moe_metadata.json")
    if best_ckpt_path.exists():
        zf.write(best_ckpt_path, arcname="remtk-juicio-moe-int8/best_moe.pt")

zip_mb = zip_bundle_path.stat().st_size / (1024 ** 2)
print("\n" + "=" * 80)
print(f"[Packaging] Created deployment bundle: {zip_bundle_path} ({zip_mb:.2f} MB)")
print("            Contents: remtk-juicio-moe-int8/model_quantized.onnx, moe_metadata.json, best_moe.pt")
print("=" * 80)

# Optional automatic browser download when running inside Google Colab
if IN_COLAB:
    try:
        from google.colab import files  # type: ignore[import-not-found]  # type: ignore[import-not-found]
        print("[Colab] Triggering browser download for 'remtk-juicio-moe-int8-bundle.zip'...")
        files.download(str(zip_bundle_path))
    except Exception as e:
        print(f"[Colab] Automatic download skipped ({e}). You can download {zip_bundle_path} from the Files panel.")